# Movie Hit-or-Flop Prediction with Pre-release Buzz
**Course project (MLDS), corrected and extended version.**

**Question.** Using only information available *before* an Indian movie is released, can we predict whether it will earn at least
twice its budget (a **Hit**), and does **pre-release buzz** (Wikipedia page views and YouTube trailer activity) add predictive value
beyond budget, language, genre and release timing?

**Data (two files shared by our team).**
* `movies_full.csv`: 4,273 movies in Hindi, Telugu, Tamil, Malayalam and Kannada, released 2015 to 2026, with budget, revenue, cast and crew,
  and Wikipedia page views in windows before release.
* `yt_features.csv`: YouTube trailer and comment statistics for 1,339 of those movies.

## Methodology in one page
| Stage | What we do | Why |
|---|---|---|
| 1. Data audit | Check ids, merge, dates, budget/revenue consistency, YouTube timing | Bad data makes every later number meaningless |
| 2. Label and cleaning | ROI = revenue / budget; Hit if ROI >= 2; every filter is logged | Nothing is dropped silently |
| 3. Feature audit | Every feature is checked for availability before release | Prevents leakage of post-release information |
| 4. Chronological split | Development period (up to 2022) and a final test period (2023 onward) | Mimics predicting the future |
| 5. Nested time-series CV | Hyperparameters are tuned inside each training window; validation predictions are collected out-of-fold | All decisions use development data only |
| 6. Buzz comparison | Base vs +Wikipedia vs +YouTube vs +both, same procedure, paired confidence intervals | The central research question |
| 7. One final test | The chosen model is frozen and scored once on the test period | An honest estimate of future performance |
| 8. Robustness | Shuffled labels, other ROI thresholds, missing buzz, language and year splits, data-source checks | Shows what the result does and does not rest on |

## Evaluation methodology (what data is used for what)
| Data | Used for | Never used for |
|---|---|---|
| Development set (released up to 2022) | EDA, hypothesis tests, clustering, feature-group and model selection, hyperparameters, decision threshold, permutation importance, robustness checks | |
| Test set (released 2023 onward) | One final evaluation of frozen models, plus one pre-declared confirmation of the buzz comparison | Choosing features, models, thresholds or hyperparameters |

**Honest expectation.** Only a few hundred movies have both budget and revenue, so every number comes with a confidence interval,
and small differences between models should not be over-read.

## 1. Setup and loading the data
The cell installs `gdown` if it is missing, downloads both files from Google Drive, and checks that each file really is the expected CSV
(a Drive error page would otherwise look like a "file"). If the download fails, Colab shows an upload button for the two files.

In [ ]:
import glob, json, subprocess, sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from IPython.display import display
from joblib import Parallel, delayed
from scipy import stats
from sklearn.base import BaseEstimator, ClassifierMixin, TransformerMixin, clone
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (adjusted_rand_score, average_precision_score, brier_score_loss, confusion_matrix,
                             mean_absolute_error, mean_squared_error, precision_recall_curve, r2_score, roc_auc_score,
                             roc_curve, silhouette_score)
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42
rng_global = np.random.default_rng(SEED)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "figure.autolayout": True})
pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_colwidth", 80)
OUT = Path("outputs")
OUT.mkdir(exist_ok=True)
results = {}            # everything reported at the end goes here and is written to outputs/results.json
T0 = time.time()
print("scikit-learn", sklearn.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)

FILES = {
    "movies_full.csv": {"id": "1ypHrvITxLgZePUA_tlKXTp-2FhfhtSlV", "must_have": ["tmdb_id", "budget", "revenue", "release_date", "language", "genres", "pv_30"]},
    "yt_features.csv": {"id": "1ScOHdRiKFXHIry03SfhOBI3Bwep2gWhZ", "must_have": ["tmdb_id", "trailer_found", "cm_pre_n"]},
}


def find_file(name):
    stem = Path(name).stem
    for folder in [".", "data", "/content", "/content/data", "../data"]:
        hits = sorted(glob.glob(f"{folder}/{stem}*.csv"))      # also matches "movies_full (1).csv" from a manual upload
        if hits:
            return Path(hits[0])
    return None


def is_valid_csv(path, must_have):
    try:
        cols = pd.read_csv(path, nrows=2).columns
    except (pd.errors.ParserError, pd.errors.EmptyDataError, UnicodeDecodeError) as err:
        print(f"  {path} is not a readable CSV: {err}")
        return False
    missing = [c for c in must_have if c not in cols]
    if missing:
        print(f"  {path} is missing expected columns {missing}")
    return not missing


def fetch_if_needed():
    for name, spec in FILES.items():
        path = find_file(name)
        if path is not None and is_valid_csv(path, spec["must_have"]):
            continue
        try:
            import gdown
        except ImportError:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gdown"])
            import gdown
        try:
            gdown.download(f"https://drive.google.com/uc?id={spec['id']}", name, quiet=True)
        except (OSError, RuntimeError, ValueError) as err:     # network, permission or quota problem; the real message is printed
            print(f"Could not download {name} from Drive: {type(err).__name__}: {err}")
    bad = [n for n, s in FILES.items() if find_file(n) is None or not is_valid_csv(find_file(n), s["must_have"])]
    if bad:
        try:
            from google.colab import files
        except ImportError:
            raise FileNotFoundError(f"Put {bad} next to this notebook (or in a data/ folder), then run this cell again.")
        print("Please upload these files:", bad)
        files.upload()
        bad = [n for n, s in FILES.items() if find_file(n) is None or not is_valid_csv(find_file(n), s["must_have"])]
        if bad:
            raise FileNotFoundError(f"Still missing or invalid after upload: {bad}")


fetch_if_needed()
raw_movies = pd.read_csv(find_file("movies_full.csv"))
raw_yt = pd.read_csv(find_file("yt_features.csv"))
print("movies_full:", raw_movies.shape, "| yt_features:", raw_yt.shape)
display(raw_movies.dtypes.rename("dtype").to_frame().T)

## 2. Data audit
Before any modelling we check the files themselves. The table below lists each check with a status:
**PASS** (nothing wrong), **WARN** (a real issue that we handle or disclose) or **INFO** (context). The original data frames `raw_movies` and `raw_yt` are never modified;
every later step works on copies, and every filter is logged.

In [ ]:
checks = []


def check(area, name, status, detail):
    checks.append({"area": area, "check": name, "status": status, "detail": detail})


# --- identifiers and merge ---
n_movies, n_yt = len(raw_movies), len(raw_yt)
check("ids", "movie tmdb_id unique and non-null", "PASS" if raw_movies.tmdb_id.is_unique and raw_movies.tmdb_id.notna().all() else "WARN",
      f"{n_movies} rows, {raw_movies.tmdb_id.duplicated().sum()} duplicate ids, {raw_movies.tmdb_id.isna().sum()} null ids")
check("ids", "YouTube tmdb_id unique and non-null", "PASS" if raw_yt.tmdb_id.is_unique and raw_yt.tmdb_id.notna().all() else "WARN",
      f"{n_yt} rows, {raw_yt.tmdb_id.duplicated().sum()} duplicate ids")
yt_orphans = int((~raw_yt.tmdb_id.isin(raw_movies.tmdb_id)).sum())
check("merge", "every YouTube id exists in the movie file", "PASS" if yt_orphans == 0 else "WARN", f"{yt_orphans} unmatched YouTube records")

merged = raw_movies.merge(raw_yt, on="tmdb_id", how="left", validate="one_to_one", indicator="yt_merge")
merged["in_yt_file"] = (merged.yt_merge == "both").astype(int)
merged = merged.drop(columns="yt_merge")
check("merge", "left merge keeps every movie exactly once", "PASS" if len(merged) == n_movies else "WARN",
      f"{n_movies} movies -> {len(merged)} rows after merge; {int(merged.in_yt_file.sum())} matched, {int((1 - merged.in_yt_file).sum())} without YouTube data")
assert len(merged) == n_movies and merged.tmdb_id.is_unique, "the merge changed the number of movies"

dup_key = raw_movies.duplicated(["name", "year", "language"], keep=False)
check("ids", "same name + year + language under different ids", "INFO", f"{int(dup_key.sum())} rows (both have no budget or revenue, so neither is used for modelling)")
check("ids", "duplicate imdb_id / wiki_title", "PASS" if not (raw_movies.imdb_id.dropna().duplicated().any() or raw_movies.wiki_title.dropna().duplicated().any()) else "WARN",
      f"imdb duplicates {int(raw_movies.imdb_id.dropna().duplicated().sum())}, wiki_title duplicates {int(raw_movies.wiki_title.dropna().duplicated().sum())}")
check("ids", "fully duplicated rows", "PASS" if not raw_movies.duplicated().any() else "WARN", f"{int(raw_movies.duplicated().sum())} duplicated rows")

# --- dates ---
merged["release_date"] = pd.to_datetime(merged.release_date, errors="coerce")
check("dates", "release_date parses", "PASS" if merged.release_date.notna().all() else "WARN", f"{int(merged.release_date.isna().sum())} unparseable dates")
check("dates", "year column agrees with release_date", "PASS" if (merged.release_date.dt.year == merged.year).all() else "WARN",
      f"{int((merged.release_date.dt.year != merged.year).sum())} mismatches; range {merged.release_date.min().date()} to {merged.release_date.max().date()}")
LAST_RELEASE = merged.release_date.max()

# --- basic types and missingness ---
miss = merged[["budget", "revenue", "runtime", "genres", "language", "pv_30", "trailer_found", "trailer_lead_days", "cm_pre_n"]].isna().mean().mul(100).round(1)
check("missing", "missing values (% of all movies)", "INFO", "; ".join(f"{k} {v}%" for k, v in miss.items()))
check("missing", "runtime equal to 0 (treated as missing)", "WARN" if (merged.runtime <= 0).any() else "PASS", f"{int((merged.runtime <= 0).sum())} movies")
merged.loc[merged.runtime <= 0, "runtime"] = np.nan        # a 0-minute film is a missing value, not a real runtime

# --- budget / revenue ---
br = merged.dropna(subset=["budget", "revenue"]).copy()
br["roi"] = br.revenue / br.budget
check("money", "budget and revenue are positive", "PASS" if ((br.budget > 0) & (br.revenue > 0)).all() else "WARN",
      f"{len(br)} movies have both; min budget ${br.budget.min():,.0f}, min revenue ${br.revenue.min():,.0f}")
br["src"] = np.where((br.budget_source == "tmdb") & (br.revenue_source == "tmdb"), "tmdb/tmdb", "has a Wikipedia value")
src_tab = br.groupby("src").agg(movies=("roi", "size"), median_budget=("budget", "median"), median_roi=("roi", "median"), hit_rate=("roi", lambda s: (s >= 2).mean())).round(3)
gap = src_tab.loc["has a Wikipedia value", "median_roi"] / src_tab.loc["tmdb/tmdb", "median_roi"]
check("money", "budget and revenue come from one consistent source", "WARN",
      f"{int((br.src != 'tmdb/tmdb').sum())} of {len(br)} movies use a Wikipedia budget or revenue; their median ROI is {gap:.2f}x that of tmdb/tmdb movies. "
      "The file has no currency column. Magnitudes look like US dollars, but a mixed currency or gross/net basis cannot be ruled out (section 17g checks this against the Wikipedia infobox text).")
print("Budget/revenue source comparison (movies with both values):")
display(src_tab)

# --- Wikipedia windows ---
wv = merged.dropna(subset=["pv_30"])
check("wikipedia", "pv_30 equals pv_30_8 + pv_7_1 (windows end before release)", "PASS" if np.isclose(wv.pv_30, wv.pv_30_8 + wv.pv_7_1).all() else "WARN",
      f"{len(wv)} movies with page views; match rate {np.isclose(wv.pv_30, wv.pv_30_8 + wv.pv_7_1).mean():.0%}")
pv_cols = ["pv_90_31", "pv_30_8", "pv_7_1", "pv_30", "pv_peak_pre", "buzz_growth"]
check("wikipedia", "no negative page views", "PASS" if (merged[pv_cols].dropna() >= 0).all().all() else "WARN", "all window counts are zero or positive")
check("wikipedia", "coverage", "INFO", f"{merged.pv_30.notna().mean():.0%} of movies have page views (movies without a Wikipedia page have none)")

# --- YouTube ---
y = merged[merged.in_yt_file == 1]
n_long = int((y.trailer_lead_days > 365).sum())
check("youtube", "lead times are not negative (trailer published before release)", "PASS" if not ((y.trailer_lead_days < 0).any() or (y.campaign_lead_days < 0).any()) else "WARN",
      f"min trailer lead {y.trailer_lead_days.min():.0f} days, min campaign lead {y.campaign_lead_days.min():.0f} days")
check("youtube", "lead times are plausible", "WARN" if n_long else "PASS",
      f"{n_long} trailers are dated more than 365 days before release (max {y.trailer_lead_days.max():.0f}); probably a wrong video match. We cap lead times at 365 days.")
yt_time_cols = [c for c in raw_yt.columns if any(k in c.lower() for k in ("date", "time", "collect", "scrape", "fetched"))]
check("youtube", "collection timestamps available to verify comment and trailer dates", "WARN" if not yt_time_cols else "PASS",
      "No collection-date column exists. Names such as cm_pre_n and trailer_lead_days say the data is pre-release, but we cannot verify this, "
      "and like counts (cm_pre_likes) are normally read after release. This is disclosed as a limitation." if not yt_time_cols else f"timestamp columns: {yt_time_cols}")
check("youtube", "coverage", "INFO", f"{int(merged.in_yt_file.sum())} movies; coverage by release year (all movies): "
      + ", ".join(f"{int(k)}: {v:.0%}" for k, v in merged.groupby('year').in_yt_file.mean().items()))

audit = pd.DataFrame(checks)
display(audit)
results["data_audit"] = audit.to_dict("records")

### What the audit means
* The merge is clean: no duplicates are created and all 4,273 movies are kept.
* **Currency and source consistency (WARN).** Some budgets or revenues come from Wikipedia and the rest from TMDB. Movies with a Wikipedia value have a much lower ROI,
  which suggests a different basis (currency or gross/net). We keep these movies, because removing them silently would be a choice too, but we (a) never use the source column as a feature and
  (b) repeat the main comparison on the clean `tmdb/tmdb` subset in the robustness section.
* **YouTube timing (WARN).** There are no collection dates in the YouTube file, so pre-release collection **cannot be verified** from the data. We treat the YouTube result as conditional on that assumption.

## 3. Cleaning and the label
**Label.** `ROI = revenue / budget`. **Hit** if ROI is at least **2.0** (box office is shared with cinemas, so 2x is a common break-even rule of thumb), otherwise **Flop**.

**Filters (each one is logged and the removed rows are shown):**
1. keep movies with both budget and revenue;
2. keep budget above $100,000 and revenue above $10,000 (smaller values are almost surely data errors);
3. keep movies old enough for the box office to be mostly final: released at least 120 days before the latest release date in the file.

Extreme ROI values (below 0.05 or above 50) are **flagged, not removed**; they are tested in the robustness section.

In [ ]:
MATURITY_DAYS = 120
CUTOFF = LAST_RELEASE - pd.Timedelta(days=MATURITY_DAYS)
flog = [("all movies in movies_full.csv", len(merged))]
s1 = merged.dropna(subset=["budget", "revenue"])
flog.append(("have both budget and revenue", len(s1)))
bad_money = s1[(s1.budget <= 1e5) | (s1.revenue <= 1e4)]
s2 = s1.drop(bad_money.index)
flog.append(("budget > $100,000 and revenue > $10,000", len(s2)))
too_recent = s2[s2.release_date > CUTOFF]
lab = s2.drop(too_recent.index).copy()
flog.append((f"released on or before {CUTOFF.date()} (latest release {LAST_RELEASE.date()} minus {MATURITY_DAYS} days)", len(lab)))
filter_log = pd.DataFrame(flog, columns=["step", "movies_left"])
filter_log["removed"] = (-filter_log.movies_left.diff()).fillna(0).astype(int)
display(filter_log)
print("Removed by the plausibility rule:")
display(bad_money[["name", "year", "language", "budget", "revenue", "budget_source", "revenue_source"]])
print(f"Removed because too recent: {len(too_recent)} movies (released {too_recent.release_date.min().date() if len(too_recent) else '-'} to {too_recent.release_date.max().date() if len(too_recent) else '-'})")

lab["roi"] = lab.revenue / lab.budget
lab["hit"] = (lab.roi >= 2.0).astype(int)
lab["suspect_roi"] = ((lab.roi < 0.05) | (lab.roi > 50)).astype(int)
lab["tmdb_only"] = ((lab.budget_source == "tmdb") & (lab.revenue_source == "tmdb")).astype(int)
lab = lab.sort_values(["release_date", "tmdb_id"]).reset_index(drop=True)
print(f"\nUsable movies: {len(lab)} | Hit rate {lab.hit.mean():.1%} | flagged extreme ROI: {int(lab.suspect_roi.sum())} | tmdb/tmdb only: {int(lab.tmdb_only.sum())}")
assert lab.tmdb_id.is_unique
results["filter_log"] = filter_log.to_dict("records")
results["usable_movies"] = int(len(lab))

## 4. Feature audit: is every input known before release?
A feature may be used only if it would be known on the day we make the prediction (before release). The table lists every feature with its reason.
The code then **asserts** that no forbidden column is in any feature list, so a mistake would stop the notebook instead of passing silently.

In [ ]:
audit_rows = [
    # feature, group, available before release?, reason / caveat
    ("log_budget", "base", "Yes", "Production budget is fixed before release. Caveat: budgets are public estimates."),
    ("runtime", "base", "Yes", "Known once the film is edited, before release. 0-minute values treated as missing."),
    ("year, month_sin, month_cos, season", "base", "Yes", "Release date is announced in advance. Season is a fixed rule (winter Dec-Feb, summer Mar-May, monsoon Jun-Sep, festive Oct-Nov), not learned from data."),
    ("language", "base", "Yes", "Fixed by the production."),
    ("genres (top 12 chosen from training data only)", "base", "Yes", "Genre tags are normally set at announcement. They could be edited later on TMDB; this cannot be checked."),
    ("l_pv_90_31, l_pv_30_8, l_pv_7_1", "Wikipedia buzz", "Yes", "Page views in windows ending 1 day before release. Verified: pv_30 = pv_30_8 + pv_7_1."),
    ("l_pv_peak_pre, l_buzz_growth", "Wikipedia buzz", "Yes (by name)", "Computed from the pre-release windows. buzz_growth tracks the ratio of late to early views (checked by correlation), but its exact formula is not documented."),
    ("buzz_per_budget", "Wikipedia buzz", "Yes", "log10(1 + pv_30) minus log10(budget). It contains the budget, so it partly repeats log_budget (see feature importance)."),
    ("trailer_found, n_promos", "YouTube", "Unverified", "Counts of videos found at collection time. If collection happened after release, later videos could be included. No date column exists."),
    ("l_trailer_lead, l_campaign_lead", "YouTube", "Yes (by definition)", "Days between the video and the release date; never negative. Capped at 365 days because larger values look like wrong matches."),
    ("l_cm_pre_n, l_cm_pre_likes", "YouTube", "Unverified", "Comments dated before release (by name). Comment likes are usually read at collection time, which may be after release, so cm_pre_likes may carry post-release information."),
    ("dir_/wri_/cast_/tech_/art_ track records (10 crew + 7 technical columns, experiment only)", "cast/crew", "Yes", "For each person, only films released at least 120 days BEFORE this movie (so their outcome was already known) are used; see section 5b, which also checks this against a brute-force recomputation. Used only in the separate experiment in section 17d."),
    ("missing-value indicators", "buzz groups", "Yes", "Created inside each training fold. Having a Wikipedia page or a YouTube record may reflect how notable a movie became, which is a selection effect, not a pure pre-release signal."),
]
feature_audit = pd.DataFrame(audit_rows, columns=["feature(s)", "group", "known before release?", "reasoning and caveat"])
display(feature_audit)

FORBIDDEN = {"rating", "vote_count", "revenue", "roi", "hit", "suspect_roi", "tmdb_only", "budget_source", "revenue_source", "imdb_id", "wiki_title", "name",
             "tmdb_id", "trailer_title", "trailer_id", "director", "writer", "screenplay", "art_director", "technical_team", "cast", "in_yt_file"}
print("Excluded on purpose: rating and vote_count (audience reactions after release), revenue, roi and the Hit label (the answer itself),")
print("budget/revenue source flags (data-origin artefacts), and ids/names. Raw cast and crew names are never features; only leak-free track-record summaries (section 5b) are used, and only in the separate experiment of section 17d.")
results["feature_audit"] = feature_audit.to_dict("records")

## 5. Deterministic feature construction
These steps use only the movie's own row (logs, calendar maths, caps). Nothing here learns from other movies, so doing it before the split cannot leak.
Everything that **learns from data** (median imputation, scaling, one-hot categories, the list of top genres) is inside the scikit-learn `Pipeline` and is fitted on training rows only.

In [ ]:
SEASON = {12: "winter", 1: "winter", 2: "winter", 3: "summer", 4: "summer", 5: "summer", 6: "monsoon", 7: "monsoon", 8: "monsoon", 9: "monsoon", 10: "festive", 11: "festive"}


def build_features(df):
    f = df.copy()
    f["log_budget"] = np.log10(f.budget)
    month = f.release_date.dt.month
    f["month"] = month
    f["month_sin"] = np.sin(2 * np.pi * month / 12)
    f["month_cos"] = np.cos(2 * np.pi * month / 12)
    f["season"] = month.map(SEASON)
    for c in ["pv_90_31", "pv_30_8", "pv_7_1", "pv_peak_pre", "buzz_growth"]:
        f["l_" + c] = np.log1p(f[c])
    f["buzz_per_budget"] = np.log10(1 + f.pv_30) - f.log_budget
    f["l_trailer_lead"] = np.log1p(f.trailer_lead_days.clip(upper=365))
    f["l_campaign_lead"] = np.log1p(f.campaign_lead_days.clip(upper=365))
    f["l_cm_pre_n"] = np.log1p(f.cm_pre_n)
    f["l_cm_pre_likes"] = np.log1p(f.cm_pre_likes)
    return f


lab = build_features(lab)
BASE_NUM = ["log_budget", "runtime", "year", "month_sin", "month_cos"]
BASE_CAT = ["language", "season"]
GENRE = ["genres"]
WIKI = ["l_pv_90_31", "l_pv_30_8", "l_pv_7_1", "l_pv_peak_pre", "l_buzz_growth", "buzz_per_budget"]
YT = ["trailer_found", "n_promos", "l_trailer_lead", "l_campaign_lead", "l_cm_pre_n", "l_cm_pre_likes"]
GROUPS = {"A: base": [], "B: base + Wikipedia": WIKI, "C: base + YouTube": YT, "D: base + Wikipedia + YouTube": WIKI + YT}
ALL_FEATURES = set(BASE_NUM + BASE_CAT + GENRE + WIKI + YT)
assert not (ALL_FEATURES & FORBIDDEN), f"forbidden column in features: {ALL_FEATURES & FORBIDDEN}"
print("Leakage guard passed. Feature groups:", {k: len(BASE_NUM + BASE_CAT + GENRE + v) for k, v in GROUPS.items()}, "raw columns")
print(f"Correlation between buzz_per_budget and log_budget (all usable movies with Wikipedia data): "
      f"{lab[['buzz_per_budget', 'log_budget']].corr().iloc[0, 1]:.2f}")

## 5b. Cast and crew track records (leak-free)
A movie's director, writers and cast are known long before release, and people with a good record may make better-performing films. The raw names cannot be used directly (every person appears in only a few films), so we summarise each person's **track record before this movie**:
* **films before** (experience): how many earlier films of that person are in the file (no outcome needed).
* **past hit rate**: the person's share of Hits among earlier films, **smoothed towards the overall Hit rate** (as if each person came with 2 average films) so one lucky film does not give a perfect record.
* **past revenue level**: the person's mean log10 revenue in earlier films (in US dollars as stored in the file; see section 17g for the currency check).

**Leakage rule.** For a movie released on day *t*, a person's outcome history may only contain films released **at least 120 days before *t*** (the same maturity rule used for labels), so the record uses only what was already known on day *t*. The overall Hit rate used for smoothing follows the same rule. Roles are tracked separately: director, writer, the five first-listed cast members (listed in billing order in the source), the technical team (composer, cinematographer, editor and so on; the source gives names without roles) and the art director. The two files have **no studio or producer column**, so those names were read separately from each movie's Wikipedia page (the *production company* and *producer* fields of its infobox, collected once on 21 November 2026 and embedded below so the notebook needs no internet access for them). Studios and producers are tracked like the other roles, but only films with an infobox record count as earlier films (the 826 labelled movies that have a Wikipedia page), so their experience counts are lower than for the other roles.
The next cell computes the features and then **proves the rule** by recomputing a sample of them with a separate brute-force method.

In [ ]:
import base64, gzip, io, re

INFOBOX_B64 = "..."     # production company, producer, budget and gross text read from each labelled movie's Wikipedia infobox (21 Nov 2026)
infobox = pd.read_csv(io.StringIO(gzip.decompress(base64.b64decode(INFOBOX_B64)).decode("utf-8")), keep_default_na=False)
_DROP = {"pvt", "ltd", "private", "limited", "films", "film", "studios", "studio", "productions", "production", "pictures", "entertainment", "entertainments", "movies", "movie",
         "motion", "creations", "international", "inc", "llp", "the", "and", "presenters", "presenter", "presents"}


def _org_list(v):
    """'T-Series Films ; Dharma Productions' -> 'tseries, dharma' (lower case, company-type words removed, duplicates removed)."""
    if not isinstance(v, str) or not v.strip():
        return np.nan
    v = v.replace("&nbsp;", " ").replace("&amp;", " and ")
    names = []
    for piece in re.split(r"\s*;\s*|\s*\*\s*|\s*,\s*|\s+/\s+", v):
        piece = re.sub(r"\(.*?\)", " ", piece.lower())
        toks = [t for t in re.sub(r"[^a-z0-9 ]", "", piece.replace("-", "")).split() if t not in _DROP]
        name = "".join(toks)
        if name and name not in names:
            names.append(name)
    return ", ".join(names) if names else np.nan


_ib = infobox.drop_duplicates("wiki_title").set_index("wiki_title")
_studio, _prod = _ib.company.map(_org_list), _ib.producer.map(_org_list)
for _df in (merged, lab):
    _df["studio"] = _df.wiki_title.map(_studio)
    _df["producer"] = _df.wiki_title.map(_prod)
print(f"Infobox records read: {len(infobox)} movies; production company found for {infobox.company.ne('').sum()}, producer found for {infobox.producer.ne('').sum()}.")
print(f"Labelled movies with a studio name: {lab.studio.notna().sum()} of {len(lab)}; with a producer name: {lab.producer.notna().sum()} of {len(lab)}.")
display(lab.studio.str.split(", ").explode().value_counts().head(8).rename("labelled movies").to_frame().T)

In [ ]:
import bisect
SMOOTH_M = 2            # a person's record counts as if they had 2 extra average films
MATURE = pd.Timedelta(days=MATURITY_DAYS)
TOP_CAST = 5


def _people(v, top=None):
    if not isinstance(v, str):
        return []
    names = [n.strip().lower() for n in v.split(",") if n.strip()]
    return names[:top] if top else names


ROLE_COLS = {"dir": ("director", None), "wri": ("writer", None), "cast": ("cast", TOP_CAST), "tech": ("technical_team", None), "art": ("art_director", None),
             "studio": ("studio", None), "prod": ("producer", None)}


def add_track_records(lab_df, all_df):
    """Track-record features for every labelled movie; each uses only films released before it (outcomes: at least MATURITY_DAYS before it)."""
    all_df = all_df[all_df.release_date.notna()]
    hist_all = {r: {} for r in ROLE_COLS}      # person -> dates of all earlier films in the file (any outcome status)
    hist_hit = {r: {} for r in ROLE_COLS}      # person -> (dates, hits) of labelled films
    hist_rev = {r: {} for r in ROLE_COLS}      # person -> (dates, log10 revenue) of films with revenue
    labd = lab_df[["tmdb_id", "release_date", "hit"]].sort_values("release_date")
    lab_dates, lab_cum = labd.release_date.values.astype("datetime64[ns]"), np.cumsum(labd.hit.values)
    for r, (col, top) in ROLE_COLS.items():
        for d, v in all_df.sort_values("release_date")[["release_date", col]].itertuples(index=False):
            for n in _people(v, top):
                hist_all[r].setdefault(n, []).append(d)
        tmp = all_df.merge(lab_df[["tmdb_id", "hit"]], on="tmdb_id", how="inner").sort_values("release_date")
        for d, v, h in tmp[["release_date", col, "hit"]].itertuples(index=False):
            for n in _people(v, top):
                hist_hit[r].setdefault(n, ([], []))[0].append(d)
                hist_hit[r][n][1].append(int(h))
        rv = all_df[all_df.revenue > 0].sort_values("release_date")
        for d, v, rev in rv[["release_date", col, "revenue"]].itertuples(index=False):
            for n in _people(v, top):
                hist_rev[r].setdefault(n, ([], []))[0].append(d)
                hist_rev[r][n][1].append(float(np.log10(rev)))
    out = []
    for row in lab_df.itertuples(index=False):
        t = row.release_date
        k = np.searchsorted(lab_dates, np.datetime64(t - MATURE), side="right")      # labelled films released at least 120 days earlier
        prior = (lab_cum[k - 1] / k) if k > 0 else 0.4
        feats = {}
        for r, (col, top) in ROLE_COLS.items():
            n_prev, rates, revs = [], [], []
            for n in _people(getattr(row, col), top):
                n_prev.append(bisect.bisect_left(hist_all[r].get(n, []), t))
                dts, hits = hist_hit[r].get(n, ([], []))
                j = bisect.bisect_right(dts, t - MATURE)
                rates.append((sum(hits[:j]) + SMOOTH_M * prior) / (j + SMOOTH_M))
                dts2, lr = hist_rev[r].get(n, ([], []))
                j2 = bisect.bisect_right(dts2, t - MATURE)
                if j2:
                    revs.append(np.mean(lr[:j2]))
            feats[f"{r}_n_prev"] = np.log1p(np.mean(n_prev)) if n_prev else np.nan
            feats[f"{r}_hit_rate"] = np.mean(rates) if rates else np.nan
            feats[f"{r}_log_rev"] = np.mean(revs) if revs else np.nan
            if r in ("cast", "tech", "studio"):
                feats[f"{r}_hit_rate_max"] = np.max(rates) if rates else np.nan
        out.append(feats)
    return pd.DataFrame(out, index=lab_df.index)


CREW = ["dir_n_prev", "dir_hit_rate", "dir_log_rev", "wri_n_prev", "wri_hit_rate", "wri_log_rev", "cast_n_prev", "cast_hit_rate", "cast_log_rev", "cast_hit_rate_max"]
TECH = ["tech_n_prev", "tech_hit_rate", "tech_log_rev", "tech_hit_rate_max", "art_n_prev", "art_hit_rate", "art_log_rev"]     # technical team (composer, cinematographer, editor ...) and art director
STUDIO = ["studio_n_prev", "studio_hit_rate", "studio_log_rev", "studio_hit_rate_max", "prod_n_prev", "prod_hit_rate", "prod_log_rev"]     # production company and producer (from Wikipedia infoboxes)
crew_df = add_track_records(lab, merged)
lab = pd.concat([lab.drop(columns=[c for c in CREW + TECH + STUDIO if c in lab.columns]), crew_df[CREW + TECH + STUDIO]], axis=1)
assert not (set(CREW + TECH + STUDIO) & FORBIDDEN)
ALL_FEATURES |= set(CREW + TECH + STUDIO)

# ---- proof: recompute the director/cast hit rates for a random sample with a different, deliberately simple method ----
rg = np.random.default_rng(SEED)
for i in rg.choice(len(lab), 60, replace=False):
    t = lab.release_date.iloc[i]
    known = lab[lab.release_date <= t - MATURE]
    prior = known.hit.mean() if len(known) else 0.4
    for r, (col, top) in ROLE_COLS.items():
        rates = []
        for n in _people(lab[col].iloc[i], top):
            mask = known[col].map(lambda v: n in _people(v, top))
            rates.append((known.hit[mask].sum() + SMOOTH_M * prior) / (mask.sum() + SMOOTH_M))
        got = lab[f"{r}_hit_rate"].iloc[i]
        assert (np.isnan(got) and not rates) or abs(got - np.mean(rates)) < 1e-9, f"track-record mismatch for row {i}, role {r}"
print("Leak-free check passed: 60 random movies, hit-rate features recomputed by brute force (only films >= 120 days earlier) and matched exactly.")
print("\nHow much information do track records carry? Share of labelled movies whose person(s) have at least one earlier film in the file:")
display(pd.DataFrame({"director": [(lab.dir_n_prev > 0).mean()], "writer": [(lab.wri_n_prev > 0).mean()], "cast (top 5, average)": [(lab.cast_n_prev > 0).mean()],
                      "technical team": [(lab.tech_n_prev > 0).mean()], "art director": [(lab.art_n_prev > 0).mean()],
                      "production company": [(lab.studio_n_prev > 0).mean()], "producer": [(lab.prod_n_prev > 0).mean()]}, index=["share"]).round(2))
print("Correlation of each track-record feature with the Hit label (all labelled movies; description only):")
display(lab[CREW + TECH + STUDIO].corrwith(lab.hit).round(3).to_frame("correlation with Hit").T)
results["crew_features"] = {"n_features": len(CREW + TECH + STUDIO), "features": CREW + TECH + STUDIO, "maturity_days": MATURITY_DAYS, "smoothing_m": SMOOTH_M, "brute_force_check": "passed (60 movies)"}

## 6. Chronological split and validation folds
**Development set:** movies released before 1 January 2023. **Test set:** movies released from 1 January 2023 on.

**Trade-off.** An earlier cut-off gives a larger and fresher test set but leaves too little training data; a later cut-off does the opposite.
The table shows the options. We keep the 2023 cut-off: it leaves a development set large enough for time-series cross-validation and a test period of about three years.
We do **not** carve a separate fixed validation year out of the development set, because single years (for example 2020 and 2021) have very few labelled movies.
Instead, validation happens **inside the development set with expanding-window `TimeSeriesSplit`**, so every validation fold is later in time than its training window.

In [ ]:
SPLIT = pd.Timestamp("2023-01-01")
dev = lab[lab.release_date < SPLIT].reset_index(drop=True)
test = lab[lab.release_date >= SPLIT].reset_index(drop=True)
assert dev.release_date.max() < test.release_date.min()

opts = []
for cut in ["2021-01-01", "2022-01-01", "2023-01-01", "2024-01-01"]:
    a, b = lab[lab.release_date < cut], lab[lab.release_date >= cut]
    opts.append({"test starts": cut, "train movies": len(a), "train hit rate": a.hit.mean(), "test movies": len(b), "test hit rate": b.hit.mean(), "test hits": int(b.hit.sum()),
                 "chosen": "<-- used" if cut == str(SPLIT.date()) else ""})
display(pd.DataFrame(opts).round(3))
print(f"Development: {len(dev)} movies, {dev.release_date.min().date()} to {dev.release_date.max().date()}, Hit rate {dev.hit.mean():.1%}")
print(f"Test:        {len(test)} movies, {test.release_date.min().date()} to {test.release_date.max().date()}, Hit rate {test.hit.mean():.1%}")
results["split"] = {"cut": str(SPLIT.date()), "dev_n": len(dev), "test_n": len(test), "dev_hit_rate": float(dev.hit.mean()), "test_hit_rate": float(test.hit.mean())}

N_OUTER = 5
outer_cv = TimeSeriesSplit(n_splits=N_OUTER)
outer_folds = list(outer_cv.split(dev))
rows = []
for k, (tr, va) in enumerate(outer_folds, 1):
    rows.append({"fold": k, "train n": len(tr), "train from": dev.release_date.iloc[tr].min().date(), "train to": dev.release_date.iloc[tr].max().date(), "train hit rate": dev.hit.iloc[tr].mean(),
                 "validation n": len(va), "validation from": dev.release_date.iloc[va].min().date(), "validation to": dev.release_date.iloc[va].max().date(), "validation hit rate": dev.hit.iloc[va].mean()})
fold_table = pd.DataFrame(rows)
print("\nOuter folds (used to compare models and feature groups). Inside each training window, hyperparameters are tuned with a further 3-fold TimeSeriesSplit.")
display(fold_table.round(3))
tr_last, _ = outer_folds[-1]
inner_rows = [{"inner fold": k, "train n": len(a), "train to": dev.release_date.iloc[tr_last[a]].max().date(), "validation n": len(b), "validation from": dev.release_date.iloc[tr_last[b]].min().date(),
               "validation to": dev.release_date.iloc[tr_last[b]].max().date()} for k, (a, b) in enumerate(TimeSeriesSplit(n_splits=3).split(tr_last), 1)]
print("Example: the 3 inner tuning folds inside outer fold 5's training window")
display(pd.DataFrame(inner_rows))
results["outer_folds"] = fold_table.astype(str).to_dict("records")

## 7. Exploring the development data
All exploration below uses the **development set only**, so nothing learned here can be influenced by the test period.
We first look at the label, then at how the hit rate moves over time. The hit rate is plotted for all years (the test years are shaded) only to describe the data; no decision uses those years.
The second line shows the hit rate for movies whose budget and revenue both come from TMDB, which separates a real change over time from a change in the data source mix.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].hist(np.log10(dev.roi), bins=40, color="#4C72B0")
for t, c in [(1, "gray"), (2, "crimson"), (3, "darkorange")]:
    ax[0].axvline(np.log10(t), color=c, ls="--", label=f"ROI = {t}")
ax[0].set(title="ROI distribution (development set)", xlabel="log10(ROI)")
ax[0].legend()
yr = lab.groupby("year").agg(n=("hit", "size"), all_movies=("hit", "mean"))
yr["tmdb/tmdb only"] = lab[lab.tmdb_only == 1].groupby("year").hit.mean()
yr[["all_movies", "tmdb/tmdb only"]].mul(100).plot(ax=ax[1], marker="o")
ax[1].axvspan(2022.5, 2026.5, color="orange", alpha=0.12, label="test years")
ax[1].set(title="Hit rate by release year (%)", ylabel="%")
ax[1].legend()
yr["n"].plot.bar(ax=ax[2], color="#8172B2")
ax[2].set(title="Usable movies per release year")
plt.savefig(OUT / "01_label_and_time_trend.png", bbox_inches="tight")
plt.show()
print("ROI summary (development set):")
display(dev.roi.describe(percentiles=[.05, .25, .5, .75, .95]).round(2).to_frame().T)
display(yr.round(3).T)
tm = lab[lab.tmdb_only == 1]
print(f"Reading the chart: the Hit rate is {dev.hit.mean():.0%} in the development set and {test.hit.mean():.0%} in the test set. "
      f"Using only tmdb/tmdb movies it is {tm[tm.release_date < SPLIT].hit.mean():.0%} and {tm[tm.release_date >= SPLIT].hit.mean():.0%}. "
      f"So the drop is mostly a real change over time and only partly caused by the data-source mix "
      f"(share of usable movies with a Wikipedia budget or revenue: {1 - dev.tmdb_only.mean():.0%} in development, {1 - test.tmdb_only.mean():.0%} in test). "
      "This label shift is a main reason why test results can be lower than validation results.")

## 8. Hypothesis tests (development set)
**Hits versus Flops.**
* **Numeric columns:** Mann-Whitney U test (budgets and page views are very skewed). We report group sizes, medians, the p-value and the **rank-biserial correlation** as effect size
  (0 means no difference, +1 means Hits are always larger, -1 always smaller).
* **Categorical columns:** chi-square test when all expected counts are adequate (at least 5 in 80% of cells and none below 1). For 2x2 tables we use **Fisher's exact test**, and for larger tables with small expected counts a **Monte Carlo permutation test**.
* **Multiple testing:** all p-values are adjusted together with Benjamini-Hochberg (controls the false discovery rate at 5%).

A small p-value says the difference is unlikely to be pure chance. It does **not** say that the feature causes success.

In [ ]:
def bh_adjust(p):
    p = np.asarray(p, float)
    order = np.argsort(p)
    adj = np.empty_like(p)
    ranked = p[order] * len(p) / (np.arange(len(p)) + 1)
    adj[order] = np.minimum(np.minimum.accumulate(ranked[::-1])[::-1], 1)
    return adj


def mannwhitney(df, col, label):
    a, b = df.loc[df.hit == 1, col].dropna(), df.loc[df.hit == 0, col].dropna()
    u, p = stats.mannwhitneyu(a, b, alternative="two-sided")
    return {"test": f"Mann-Whitney: {label}", "n_hit": len(a), "n_flop": len(b), "median_hit": a.median(), "median_flop": b.median(), "effect": 2 * u / (len(a) * len(b)) - 1, "p": p, "method": "Mann-Whitney U"}


def chi2_stat(counts):
    exp = counts.sum(1, keepdims=True) * counts.sum(0, keepdims=True) / counts.sum()
    return ((counts - exp) ** 2 / exp).sum()


def categorical_test(df, col, label, n_perm=10000, seed=SEED):
    d = df[[col, "hit"]].dropna()
    ct = pd.crosstab(d[col], d.hit)
    exp = stats.contingency.expected_freq(ct.values)
    if ct.shape == (2, 2):
        p, method = stats.fisher_exact(ct.values)[1], "Fisher exact"
    elif (exp < 5).mean() > 0.2 or exp.min() < 1:
        codes = pd.factorize(d[col])[0]
        y = d.hit.values
        k = codes.max() + 1
        obs = chi2_stat(np.bincount(codes * 2 + y, minlength=2 * k).reshape(k, 2).astype(float))
        rg = np.random.default_rng(seed)
        sims = np.array([chi2_stat(np.bincount(codes * 2 + rg.permutation(y), minlength=2 * k).reshape(k, 2).astype(float)) for _ in range(n_perm)])
        p, method = (1 + (sims >= obs).sum()) / (n_perm + 1), "Monte Carlo chi-square (small expected counts)"
    else:
        p, method = stats.chi2_contingency(ct.values)[1], "chi-square"
    rate = d.groupby(col).hit.mean()
    return {"test": f"{method.split(' (')[0]}: {label} vs Hit", "n_hit": int(ct[1].sum()), "n_flop": int(ct[0].sum()), "median_hit": np.nan, "median_flop": np.nan, "effect": np.nan, "p": p,
            "method": method, "min_expected": exp.min(), "hit_rate_range": f"{rate.min():.2f} to {rate.max():.2f}"}


dev_t = dev.copy()
dev_t["has_wikipedia"] = dev_t.pv_30.notna().astype(int)
dev_t["trailer_found_yt"] = dev_t.trailer_found
tests = [mannwhitney(dev_t, "log_budget", "log10 budget"), mannwhitney(dev_t, "runtime", "runtime (min)"),
         mannwhitney(dev_t, "pv_30", "Wikipedia views, 30 days before release"), mannwhitney(dev_t, "pv_90_31", "Wikipedia views, 90-31 days before"),
         mannwhitney(dev_t, "buzz_per_budget", "buzz per budget"), mannwhitney(dev_t, "cm_pre_n", "YouTube comments before release"),
         mannwhitney(dev_t, "trailer_lead_days", "trailer lead time (days)"), mannwhitney(dev_t, "n_promos", "number of promo videos"),
         categorical_test(dev_t, "language", "language"), categorical_test(dev_t, "season", "season"), categorical_test(dev_t, "month", "release month"),
         categorical_test(dev_t, "year", "release year"), categorical_test(dev_t, "has_wikipedia", "has a Wikipedia page-view record"),
         categorical_test(dev_t, "trailer_found_yt", "trailer found (YouTube movies only)")]
tt = pd.DataFrame(tests)
tt["p_adjusted_BH"] = bh_adjust(tt.p.values)
tt["significant_at_5pct"] = tt.p_adjusted_BH < 0.05
display(tt[["test", "n_hit", "n_flop", "median_hit", "median_flop", "effect", "p", "p_adjusted_BH", "significant_at_5pct", "method"]].round(4))
results["hypothesis_tests"] = tt.round(5).replace({np.nan: None}).to_dict("records")

sig = tt[tt.significant_at_5pct]
print(f"{len(sig)} of {len(tt)} tests are significant after Benjamini-Hochberg correction:")
for _, r in sig.iterrows():
    extra = f"effect size {r.effect:+.2f}" if pd.notna(r.effect) else f"hit rates range {r.hit_rate_range}"
    print(f"  - {r.test} (adjusted p = {r.p_adjusted_BH:.4f}, {extra})")
print("A significant association is not a cause: for example, low-budget movies being Hits more often does not mean lowering a budget will create a Hit.")

In [ ]:
lang = dev.groupby("language").agg(movies=("hit", "size"), hits=("hit", "sum"), hit_rate=("hit", "mean"), median_budget_musd=("budget", lambda s: s.median() / 1e6)).sort_values("hit_rate")
lang["95% CI (Wilson)"] = [f"{lo:.2f} to {hi:.2f}" for lo, hi in
                           [stats.binomtest(int(h), int(n)).proportion_ci(method="wilson") for h, n in zip(lang.hits, lang.movies)]]
print("Hit rate by language (development set):")
display(lang.round(3))
results["hit_rate_by_language_dev"] = lang.reset_index().round(4).to_dict("records")

num_cols = ["log_budget", "runtime", "year", "l_pv_90_31", "l_pv_30_8", "l_pv_7_1", "l_buzz_growth", "buzz_per_budget", "l_cm_pre_n", "l_cm_pre_likes", "n_promos", "l_trailer_lead"]
dev_c = dev.assign(log10_roi=np.log10(dev.roi))
sp = []
for c in num_cols:
    sub = dev_c[[c, "log10_roi"]].dropna()
    rho, p = stats.spearmanr(sub[c], sub.log10_roi)
    sp.append({"feature": c, "n": len(sub), "spearman_rho": rho, "p": p})
sp = pd.DataFrame(sp)
sp["p_adjusted_BH"] = bh_adjust(sp.p.values)
print("Spearman correlation of each pre-release feature with log10(ROI), development set:")
display(sp.round(4))
results["spearman_vs_log_roi"] = sp.round(5).to_dict("records")
corr = dev_c[num_cols + ["log10_roi"]].corr(method="spearman")
plt.figure(figsize=(9, 7))
sns.heatmap(corr, cmap="coolwarm", center=0, annot=True, fmt=".2f", annot_kws={"size": 7}, cbar_kws={"shrink": 0.7})
plt.title("Spearman correlations (development set)")
plt.savefig(OUT / "02_correlations.png", bbox_inches="tight")
plt.show()

## 9. Clustering (exploratory)
K-means on three standardised columns: log budget, runtime and log Wikipedia views (movies with page-view data). We
1. try K = 2 to 8 and compare **silhouette scores**,
2. check **cluster sizes** and profiles,
3. test **stability**: re-cluster 30 bootstrap resamples and compare with the main solution using the adjusted Rand index (1 = identical grouping),
4. show the clusters in a **PCA** view.

Whether clusters differ in hit rate is **exploratory**: clusters are built without the label, so a difference is a description of the groups, not an explanation of success.

In [ ]:
cl = dev.dropna(subset=["pv_30", "runtime"]).copy()
cl["log_pv30"] = np.log10(1 + cl.pv_30)
cl_cols = ["log_budget", "runtime", "log_pv30"]
Xc = StandardScaler().fit_transform(cl[cl_cols])
sil = {k: silhouette_score(Xc, KMeans(k, n_init=10, random_state=SEED).fit_predict(Xc)) for k in range(2, 9)}
best_k = max(sil, key=sil.get)
km = KMeans(best_k, n_init=10, random_state=SEED).fit(Xc)
cl["cluster"] = km.labels_
rg = np.random.default_rng(SEED)
ari = []
for _ in range(30):
    idx = rg.integers(0, len(Xc), len(Xc))
    boot = KMeans(best_k, n_init=5, random_state=int(rg.integers(1e6))).fit(Xc[idx])
    ari.append(adjusted_rand_score(km.labels_, boot.predict(Xc)))
prof = cl.groupby("cluster").agg(movies=("hit", "size"), hit_rate=("hit", "mean"), median_budget_musd=("budget", lambda s: s.median() / 1e6), median_pageviews_30d=("pv_30", "median"),
                                 median_runtime=("runtime", "median"), share_hindi=("language", lambda s: (s == "hi").mean())).round(2)
p_cl = stats.chi2_contingency(pd.crosstab(cl.cluster, cl.hit))[1]
print("Silhouette by K:", {k: round(v, 3) for k, v in sil.items()}, "-> best K =", best_k)
print(f"Bootstrap stability (adjusted Rand index): mean {np.mean(ari):.2f}, min {np.min(ari):.2f}")
display(prof)
print(f"Hit rate differs between clusters: chi-square p = {p_cl:.4f} (exploratory).")
quality = "weak" if sil[best_k] < 0.35 else ("moderate" if sil[best_k] < 0.5 else "strong")
stability = "stable" if np.mean(ari) >= 0.8 else ("moderately stable" if np.mean(ari) >= 0.6 else "unstable")
print(f"Interpretation: the cluster structure is {quality} (silhouette {sil[best_k]:.2f}) and {stability}. "
      "Treat the groups as a rough description (for example, big-budget, high-attention films versus small-budget films), not as fixed categories.")
results["clustering"] = {"k": int(best_k), "silhouette": {int(k): float(v) for k, v in sil.items()}, "stability_ari_mean": float(np.mean(ari)), "chi2_p": float(p_cl), "profile": prof.reset_index().to_dict("records")}
pc = PCA(2, random_state=SEED)
Z = pc.fit_transform(Xc)
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
ax[0].plot(list(sil), list(sil.values()), marker="o")
ax[0].axvline(best_k, color="gray", ls="--")
ax[0].set(title="Silhouette score by K", xlabel="K", ylabel="silhouette")
sns.scatterplot(x=Z[:, 0], y=Z[:, 1], hue=cl.cluster.astype(str), style=cl.hit.map({0: "Flop", 1: "Hit"}), palette="Set2", s=24, ax=ax[1])
ax[1].set(title=f"Clusters in PCA space (explains {pc.explained_variance_ratio_.sum():.0%} of variance)", xlabel="PC1", ylabel="PC2")
plt.savefig(OUT / "03_clusters.png", bbox_inches="tight")
plt.show()

## 10. Models and leakage-safe pipelines
Three classifiers, each wrapped in one scikit-learn **Pipeline**, so every learned step is fitted on the training rows of the current fold only:
1. **Logistic regression**: a linear baseline; needs scaled inputs.
2. **Random forest**: many decision trees averaged; handles non-linear effects and interactions.
3. **HistGradientBoosting**: trees added one after another to correct earlier errors; usually the strongest on tabular data.

**Inside the pipeline (all fitted on training rows only):** median imputation with missing-value indicators, scaling (logistic regression only), one-hot encoding of language and season,
and a custom `GenreTopK` step that **chooses the 12 most frequent genres from the training rows** and encodes every movie with them. This fixes the earlier problem where the top genres were chosen using all movies.

**Hyperparameters** are tuned by `GridSearchCV` with a chronological `TimeSeriesSplit` (scoring: ROC-AUC). A **DummyClassifier** that always predicts the training hit rate is the no-skill baseline.

**Nested validation.** For each outer fold (train on the past, validate on the next period) the grid search runs only inside that fold's training window.
The outer validation predictions are therefore honest "future" predictions made without seeing that period. All 12 combinations (3 models x 4 feature groups) are evaluated the same way.

In [ ]:
class GenreTopK(BaseEstimator, TransformerMixin):
    """Encode the k most frequent genres *of the training rows*; unseen genres are ignored at prediction time."""

    def __init__(self, k=12):
        self.k = k

    @staticmethod
    def _split(X):
        s = pd.Series(np.asarray(X, dtype=object).ravel()).fillna("")
        return [[g.strip() for g in str(t).split(",") if g.strip()] for t in s]

    def fit(self, X, y=None):
        counts = pd.Series([g for L in self._split(X) for g in L]).value_counts()
        self.genres_ = list(counts.index[: self.k])
        return self

    def transform(self, X):
        return np.array([[float(g in L) for g in self.genres_] for L in self._split(X)])

    def get_feature_names_out(self, input_features=None):
        return np.array([f"genre_{g}" for g in self.genres_])


CREW_GROUPS = {"E: base + cast/crew": CREW, "F: base + Wikipedia + YouTube + cast/crew": WIKI + YT + CREW,
               "G: base + technical team": TECH, "H: base + Wikipedia + YouTube + technical team": WIKI + YT + TECH,
               "I: base + production company/producer": STUDIO, "J: base + Wikipedia + YouTube + production company/producer": WIKI + YT + STUDIO}                 # used only in the cast/crew experiment (section 17d)
ALL_GROUPS = {**GROUPS, "B-: base + Wikipedia without buzz_per_budget": [c for c in WIKI if c != "buzz_per_budget"], **CREW_GROUPS,
              "D-: base + Wikipedia + YouTube without comment likes": WIKI + [c for c in YT if c != "l_cm_pre_likes"]}   # extra groups used only outside the main comparison


def make_preprocessor(group, scale):
    cols = ALL_GROUPS[group]
    wiki_cols, yt_cols, crew_cols = [c for c in cols if c in WIKI], [c for c in cols if c in YT], [c for c in cols if c in CREW + TECH + STUDIO]

    def num_pipe(indicator):
        steps = [("imp", SimpleImputer(strategy="median", add_indicator=indicator, keep_empty_features=True))]
        return Pipeline(steps + ([("sc", StandardScaler())] if scale else []))

    parts = [("num", num_pipe(False), BASE_NUM), ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), BASE_CAT), ("genre", GenreTopK(12), GENRE)]
    if wiki_cols:
        parts.append(("wiki", num_pipe(True), wiki_cols))
    if yt_cols:
        parts.append(("yt", num_pipe(True), yt_cols))
    if crew_cols:
        parts.append(("crew", num_pipe(True), crew_cols))
    return ColumnTransformer(parts, remainder="drop", sparse_threshold=0)


MODELS = {
    "Logistic regression": (lambda: LogisticRegression(max_iter=5000), {"model__C": [0.03, 0.1, 0.3, 1.0]}, True),
    "Random forest": (lambda: RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=1), {"model__min_samples_leaf": [3, 8, 15], "model__max_features": ["sqrt", 0.5]}, False),
    "HistGradientBoosting": (lambda: HistGradientBoostingClassifier(random_state=SEED, max_iter=150), {"model__learning_rate": [0.03, 0.08], "model__max_depth": [2, 3], "model__min_samples_leaf": [15, 30]}, False),
}


def make_pipe(model_key, group, params=None):
    est, _, scale = MODELS[model_key]
    pipe = Pipeline([("prep", make_preprocessor(group, scale)), ("model", est())])
    return pipe.set_params(**params) if params else pipe


def tune(X, y, model_key, group, n_inner=3):
    gs = GridSearchCV(make_pipe(model_key, group), MODELS[model_key][1], scoring="roc_auc", cv=TimeSeriesSplit(n_splits=n_inner), n_jobs=1, refit=True)
    return gs.fit(X, y)


# ---- small statistics helpers (fast AUC, bootstrap intervals) ----
def auc_fast(y, p):
    y = np.asarray(y)
    n1 = y.sum()
    n0 = len(y) - n1
    return (stats.rankdata(p)[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)


def boot_auc_ci(y, p, n=2000, seed=SEED):
    y, p = np.asarray(y), np.asarray(p)
    rg = np.random.default_rng(seed)
    vals = []
    while len(vals) < n:
        i = rg.integers(0, len(y), len(y))
        if 0 < y[i].sum() < len(i):
            vals.append(auc_fast(y[i], p[i]))
    return tuple(np.percentile(vals, [2.5, 97.5]))


def paired_boot(y, p_new, p_old, n=2000, seed=SEED):
    """Paired bootstrap of the AUC difference (new minus old) on the same resampled movies."""
    y, p_new, p_old = np.asarray(y), np.asarray(p_new), np.asarray(p_old)
    rg = np.random.default_rng(seed)
    d = []
    while len(d) < n:
        i = rg.integers(0, len(y), len(y))
        if 0 < y[i].sum() < len(i):
            d.append(auc_fast(y[i], p_new[i]) - auc_fast(y[i], p_old[i]))
    d = np.array(d)
    p = min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean()))
    return auc_fast(y, p_new) - auc_fast(y, p_old), *np.percentile(d, [2.5, 97.5]), max(p, 1 / n)


def clf_metrics(y, p, thr):
    y = np.asarray(y)
    pred = (np.asarray(p) >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    div = lambda a, b: a / b if b else np.nan
    prec, rec = div(tp, tp + fp), div(tp, tp + fn)
    return {"auc": auc_fast(y, p), "accuracy": (tp + tn) / len(y), "precision": prec, "hit_recall": rec, "flop_recall": div(tn, tn + fp),
            "f1": div(2 * prec * rec, prec + rec) if (prec == prec and rec == rec) else np.nan, "brier": brier_score_loss(y, p), "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)}


def youden_threshold(y, p):
    fpr, tpr, thr = roc_curve(y, p)
    return float(thr[np.argmax(tpr - fpr)])


y_dev, y_test = dev.hit.values, test.hit.values
val_idx = np.concatenate([va for _, va in outer_folds])          # positions in `dev` that receive an out-of-fold prediction
print(f"{len(val_idx)} of {len(dev)} development movies receive out-of-fold validation predictions (the earliest {len(dev) - len(val_idx)} only ever train).")

In [ ]:
def fold_task(model_key, group, k, tr, va):
    gs = tune(dev.iloc[tr], y_dev[tr], model_key, group)
    return model_key, group, k, va, gs.predict_proba(dev.iloc[va])[:, 1], gs.best_params_, gs.best_estimator_


t_start = time.time()
jobs = [(m, g, k, tr, va) for m in MODELS for g in GROUPS for k, (tr, va) in enumerate(outer_folds)]
out = Parallel(n_jobs=-1)(delayed(fold_task)(*j) for j in jobs)
oof, fold_models, fold_params = {}, {}, {}
for model_key, group, k, va, p, params, est in out:
    oof.setdefault((model_key, group), np.full(len(dev), np.nan))[va] = p
    fold_models.setdefault((model_key, group), {})[k] = (est, va)
    fold_params.setdefault((model_key, group), {})[k] = params
print(f"Nested cross-validation finished in {time.time() - t_start:.0f} s ({len(jobs)} outer fits, each with an inner grid search).")

rows = []
for (model_key, group), p_all in oof.items():
    fa = [auc_fast(y_dev[va], p_all[va]) for _, va in outer_folds]
    lo, hi = boot_auc_ci(y_dev[val_idx], p_all[val_idx])
    rows.append({"feature group": group, "model": model_key, "cv_auc_mean": np.mean(fa), "cv_auc_sd_over_folds": np.std(fa, ddof=1), "pooled_oof_auc": auc_fast(y_dev[val_idx], p_all[val_idx]),
                 "ci_low": lo, "ci_high": hi, "brier": brier_score_loss(y_dev[val_idx], p_all[val_idx]), "avg_precision": average_precision_score(y_dev[val_idx], p_all[val_idx]),
                 "fold_aucs": [round(a, 3) for a in fa]})
cv_table = pd.DataFrame(rows)
base_rate = y_dev[val_idx].mean()
dummy_row = {"feature group": "(none)", "model": "DummyClassifier (training hit rate)", "cv_auc_mean": 0.5, "cv_auc_sd_over_folds": 0.0, "pooled_oof_auc": 0.5, "ci_low": 0.5, "ci_high": 0.5,
             "brier": np.mean([brier_score_loss(y_dev[va], np.full(len(va), y_dev[tr].mean())) for tr, va in outer_folds]), "avg_precision": base_rate, "fold_aucs": [0.5] * N_OUTER}
cv_table = pd.concat([pd.DataFrame([dummy_row]), cv_table], ignore_index=True)
print("Validation results (out-of-fold, development period). cv_auc_mean is the average over the 5 chronological folds and is the number used for selection; the interval is a bootstrap 95% CI on the pooled predictions.")
display(cv_table.round(3))
print(f"Class balance of the validation movies: {base_rate:.1%} Hits. The DummyClassifier gets AUC 0.5 by construction, and always predicting Flop would give accuracy {1 - base_rate:.1%}.")
results["validation_table"] = cv_table.round(4).to_dict("records")

## 11. The central question: does pre-release buzz help?
For each model type we compare feature groups **B, C and D against the base group A on the same validation movies and folds**.
* **AUC improvement:** the pooled out-of-fold AUC of the richer group minus that of the base group.
* **Paired bootstrap 95% interval:** resample the validation movies, recompute both AUCs on the *same* resample, and take the difference. If the interval contains 0, the improvement could be luck.
* **Fold consistency:** in how many of the 5 chronological folds the richer group is better.
* **Multiple comparisons:** with 9 comparisons, we adjust the bootstrap p-values with Benjamini-Hochberg.

We call an improvement **convincing** only if the interval excludes 0 **and** the adjusted p-value is below 0.05. The test set is **not** used here.

In [ ]:
base_g = "A: base"
pair_rows = []
for model_key in MODELS:
    for g in list(GROUPS)[1:]:
        d, lo, hi, p = paired_boot(y_dev[val_idx], oof[(model_key, g)][val_idx], oof[(model_key, base_g)][val_idx])
        better = sum(auc_fast(y_dev[va], oof[(model_key, g)][va]) > auc_fast(y_dev[va], oof[(model_key, base_g)][va]) for _, va in outer_folds)
        pair_rows.append({"model": model_key, "added to base": g.split(": ")[1].replace("base + ", ""), "AUC gain": d, "ci_low": lo, "ci_high": hi, "p_bootstrap": p, "folds_better (of 5)": int(better)})
pairs = pd.DataFrame(pair_rows)
pairs["p_adjusted_BH"] = bh_adjust(pairs.p_bootstrap.values)
pairs["convincing"] = (pairs.ci_low > 0) & (pairs.p_adjusted_BH < 0.05)
display(pairs.round(3))
results["paired_gain_validation"] = pairs.round(4).to_dict("records")

fig, ax = plt.subplots(figsize=(9, 4.2))
lab_y = [f"{r.model} | {r['added to base']}" for _, r in pairs.iterrows()][::-1]
pr_ = pairs.iloc[::-1]
ax.errorbar(pr_["AUC gain"], range(len(pr_)), xerr=[pr_["AUC gain"] - pr_.ci_low, pr_.ci_high - pr_["AUC gain"]], fmt="o", color="#4C72B0", capsize=3)
ax.axvline(0, color="crimson", ls="--")
ax.set_yticks(range(len(pr_)))
ax.set_yticklabels(lab_y, fontsize=8)
ax.set(title="AUC improvement over the base features (validation, paired 95% interval)", xlabel="AUC gain (right of the red line = better than base)")
plt.savefig(OUT / "04_buzz_gain_validation.png", bbox_inches="tight")
plt.show()

by_source = pairs.groupby("added to base").agg(mean_gain=("AUC gain", "mean"), convincing=("convincing", "sum"), n_models=("model", "size"), folds_better=("folds_better (of 5)", "mean"))
display(by_source.round(3))
results["buzz_gain_by_source"] = by_source.reset_index().round(4).to_dict("records")
n_conv = int(pairs.convincing.sum())
print(f"Summary: {n_conv} of {len(pairs)} comparisons are convincing. "
      + ("There is no convincing evidence that buzz improves prediction on the validation data." if n_conv == 0 else
         "Where buzz helps, the table shows which model and feature source; read the other rows as inconclusive."))

## 12. Choosing the final model (validation data only)
The final model is the combination with the **highest mean cross-validated AUC** over the chronological folds. For each model type we also record its own best feature group, so that all three models can be reported fairly.
If several combinations are within the fold-to-fold noise, the choice is not very meaningful; the table above shows how close they are.

In [ ]:
real = cv_table[cv_table.model != "DummyClassifier (training hit rate)"].copy()
best_per_model = real.loc[real.groupby("model").cv_auc_mean.idxmax()].set_index("model")
chosen_row = real.loc[real.cv_auc_mean.idxmax()]
CH_MODEL, CH_GROUP = chosen_row.model, chosen_row["feature group"]
second = real.sort_values("cv_auc_mean", ascending=False).iloc[1]
print(f"Chosen by cross-validation: {CH_MODEL} on '{CH_GROUP}' (mean CV AUC {chosen_row.cv_auc_mean:.3f} +/- {chosen_row.cv_auc_sd_over_folds:.3f} over folds).")
print(f"Runner-up: {second.model} on '{second['feature group']}' ({second.cv_auc_mean:.3f}). Difference to the winner: {chosen_row.cv_auc_mean - second.cv_auc_mean:.3f}, "
      f"smaller than the fold-to-fold standard deviation ({chosen_row.cv_auc_sd_over_folds:.3f}) = " + ("the two are effectively tied." if chosen_row.cv_auc_mean - second.cv_auc_mean < chosen_row.cv_auc_sd_over_folds else "a clear gap."))
print("\nBest feature group for each model type (by cross-validation):")
display(best_per_model[["feature group", "cv_auc_mean", "cv_auc_sd_over_folds", "pooled_oof_auc", "ci_low", "ci_high"]].round(3))
results["chosen_by_cv"] = {"model": CH_MODEL, "group": CH_GROUP, "cv_auc_mean": float(chosen_row.cv_auc_mean), "cv_auc_sd": float(chosen_row.cv_auc_sd_over_folds)}

# Are the three model types really different? Paired bootstrap on validation predictions, each model with its own best feature group.
names = list(MODELS)
mm_rows = []
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        a, b = names[i], names[j]
        pa, pb = oof[(a, best_per_model.loc[a, "feature group"])][val_idx], oof[(b, best_per_model.loc[b, "feature group"])][val_idx]
        d, lo, hi, p = paired_boot(y_dev[val_idx], pa, pb)
        mm_rows.append({"model 1": a, "model 2": b, "AUC difference (1 minus 2)": d, "ci_low": lo, "ci_high": hi, "p_bootstrap": p})
model_pairs = pd.DataFrame(mm_rows)
model_pairs["p_adjusted_BH"] = bh_adjust(model_pairs.p_bootstrap.values)
model_pairs["distinguishable"] = (model_pairs.ci_low > 0) | (model_pairs.ci_high < 0)
print("\nPaired comparison of the model types on validation predictions:")
display(model_pairs.round(3))
results["model_pairs_validation"] = model_pairs.round(4).to_dict("records")

## 13. Freezing the models and the one final test
Now, and only now, the test period is used. Nothing in this section feeds back into any choice.
1. For each model type, the feature group chosen above is **refitted on the whole development set** (hyperparameters tuned with `TimeSeriesSplit` on development data only).
2. The **decision threshold** for each model is chosen by **Youden's J** (the point on the ROC curve that maximises Hit recall plus Flop recall minus 1) from the **out-of-fold validation predictions**, never from test labels.
3. Each frozen model scores the test movies **once**. Intervals are bootstrap 95% confidence intervals over test movies.

Two DummyClassifier baselines are scored the same way: `prior` (always predicts the training majority class, Flop) and `stratified` (guesses randomly in the training proportions).

In [ ]:
final_cache = {}


def final_fit(model_key, group):
    if (model_key, group) not in final_cache:
        final_cache[(model_key, group)] = tune(dev, y_dev, model_key, group, n_inner=N_OUTER)   # chronological 5-fold search on development data only
    return final_cache[(model_key, group)]


def metric_ci(y, p, thr, n=1000, seed=SEED):
    y, p = np.asarray(y), np.asarray(p)
    rg = np.random.default_rng(seed)
    keys = ["auc", "accuracy", "precision", "hit_recall", "flop_recall", "f1", "brier"]
    vals = {k: [] for k in keys}
    for _ in range(n):
        i = rg.integers(0, len(y), len(y))
        yy, pp = y[i], p[i]
        if not 0 < yy.sum() < len(yy):
            continue
        pred = pp >= thr
        tp, fp = (pred & (yy == 1)).sum(), (pred & (yy == 0)).sum()
        fn, tn = (~pred & (yy == 1)).sum(), (~pred & (yy == 0)).sum()
        prec = tp / (tp + fp) if tp + fp else np.nan
        rec = tp / (tp + fn)
        vals["auc"].append(auc_fast(yy, pp))
        vals["accuracy"].append((tp + tn) / len(yy))
        vals["precision"].append(prec)
        vals["hit_recall"].append(rec)
        vals["flop_recall"].append(tn / (tn + fp))
        vals["f1"].append(2 * prec * rec / (prec + rec) if (prec == prec and prec + rec > 0) else np.nan)
        vals["brier"].append(np.mean((pp - yy) ** 2))
    return {k: tuple(np.nanpercentile(v, [2.5, 97.5])) for k, v in vals.items()}


fmt = lambda v, ci: f"{v:.3f} [{ci[0]:.3f}, {ci[1]:.3f}]" if v == v else "n/a"
test_preds, final_rows, test_metrics = {}, [], {}
for model_key in MODELS:
    grp = best_per_model.loc[model_key, "feature group"]
    gs = final_fit(model_key, grp)
    p = gs.predict_proba(test)[:, 1]
    thr = youden_threshold(y_dev[val_idx], oof[(model_key, grp)][val_idx])
    m, ci = clf_metrics(y_test, p, thr), metric_ci(y_test, p, thr)
    test_preds[model_key], test_metrics[model_key] = (p, thr, grp), m
    final_rows.append({"model": model_key, "feature group": grp, "threshold": thr, **{k: fmt(m[k], ci[k]) for k in ["auc", "accuracy", "precision", "hit_recall", "flop_recall", "f1", "brier"]},
                       "TN": m["tn"], "FP": m["fp"], "FN": m["fn"], "TP": m["tp"]})
    results.setdefault("final_test", {})[model_key] = {"group": grp, "threshold": thr, **m, "ci": {k: list(v) for k, v in ci.items()}, "best_params": {k: (v.item() if hasattr(v, "item") else v) for k, v in gs.best_params_.items()}}

for strat in ["prior", "stratified"]:
    dm = DummyClassifier(strategy=strat, random_state=SEED).fit(np.zeros((len(dev), 1)), y_dev)     # a dummy ignores the features
    p = dm.predict_proba(np.zeros((len(test), 1)))[:, 1]
    pred_hard = dm.predict(np.zeros((len(test), 1)))
    tn, fp, fn, tp = confusion_matrix(y_test, pred_hard, labels=[0, 1]).ravel()
    prec, rec = (tp / (tp + fp) if tp + fp else np.nan), tp / (tp + fn)
    final_rows.append({"model": f"DummyClassifier ({strat})", "feature group": "(none)", "threshold": np.nan, "auc": "0.500 (no skill)", "accuracy": f"{(tp + tn) / len(y_test):.3f}",
                       "precision": f"{prec:.3f}" if prec == prec else "n/a (never predicts Hit)", "hit_recall": f"{rec:.3f}", "flop_recall": f"{tn / (tn + fp):.3f}", "f1": f"{2 * prec * rec / (prec + rec):.3f}" if prec == prec and prec + rec > 0 else "0.000 / n/a",
                       "brier": f"{brier_score_loss(y_test, p):.3f}", "TN": tn, "FP": fp, "FN": fn, "TP": tp})
final_table = pd.DataFrame(final_rows)
print(f"FINAL TEST RESULTS ({len(test)} movies released {test.release_date.min().date()} to {test.release_date.max().date()}, Hit rate {y_test.mean():.1%}). Format: value [95% CI].")
display(final_table)
results["final_table"] = final_table.to_dict("records")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4.6))
colors = {"Logistic regression": "#4C72B0", "Random forest": "#55A868", "HistGradientBoosting": "#C44E52"}
for mk, (p, thr, grp) in test_preds.items():
    f_, t_, _ = roc_curve(y_test, p)
    ax[0].plot(f_, t_, lw=2, color=colors[mk], label=f"{mk} (AUC {auc_fast(y_test, p):.2f})")
    pr, rc, _ = precision_recall_curve(y_test, p)
    ax[1].plot(rc, pr, lw=2, color=colors[mk], label=f"{mk} (AP {average_precision_score(y_test, p):.2f})")
ax[0].plot([0, 1], [0, 1], "k--", label="Dummy (AUC 0.50)")
ax[0].set(title="ROC curves, test period", xlabel="False positive rate (Flops called Hit)", ylabel="True positive rate (Hit recall)")
ax[0].legend(loc="lower right", fontsize=8)
ax[1].axhline(y_test.mean(), color="k", ls="--", label=f"Dummy (precision = Hit rate {y_test.mean():.2f})")
ax[1].set(title="Precision-recall curves, test period", xlabel="Recall (Hits found)", ylabel="Precision", ylim=(0, 1.02))
ax[1].legend(loc="upper right", fontsize=8)
cmat = np.array([[test_metrics[CH_MODEL]["tn"], test_metrics[CH_MODEL]["fp"]], [test_metrics[CH_MODEL]["fn"], test_metrics[CH_MODEL]["tp"]]])
sns.heatmap(cmat, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax[2], xticklabels=["pred Flop", "pred Hit"], yticklabels=["actual Flop", "actual Hit"])
ax[2].set_title(f"Confusion matrix: {CH_MODEL}\n(threshold {test_preds[CH_MODEL][1]:.2f}; chosen model)")
plt.savefig(OUT / "05_test_roc_pr_confusion.png", bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(1, 3, figsize=(14, 3.8))
for a, (mk, m) in zip(ax, test_metrics.items()):
    sns.heatmap([[m["tn"], m["fp"]], [m["fn"], m["tp"]]], annot=True, fmt="d", cmap="Blues", cbar=False, ax=a, xticklabels=["pred Flop", "pred Hit"], yticklabels=["actual Flop", "actual Hit"])
    a.set_title(f"{mk}\n({test_preds[mk][2].split(':')[0]}, threshold {test_preds[mk][1]:.2f})", fontsize=10)
plt.savefig(OUT / "06_test_confusion_matrices.png", bbox_inches="tight")
plt.show()

## 14. Threshold choice and Hit recall
The default Youden threshold balances Hit recall and Flop recall. If missing a Hit is costly, a lower threshold finds more Hits but also calls more Flops "Hit" (false positives).
We pick several candidate thresholds **using the out-of-fold validation predictions of the chosen model only**, then apply them frozen to the test movies to see the real trade-off.
The rule "Hit recall at least X" chooses the highest threshold that still finds X of the validation Hits, which keeps precision as high as possible for that recall.

In [ ]:
p_val, y_val = oof[(CH_MODEL, CH_GROUP)][val_idx], y_dev[val_idx]
p_te = final_fit(CH_MODEL, CH_GROUP).predict_proba(test)[:, 1]
pr_v, rc_v, th_v = precision_recall_curve(y_val, p_val)
f1_v = 2 * pr_v[:-1] * rc_v[:-1] / np.maximum(pr_v[:-1] + rc_v[:-1], 1e-9)
cands = {"Youden J (default)": youden_threshold(y_val, p_val), "maximum F1": float(th_v[np.argmax(f1_v)])}
for target in (0.70, 0.80):
    ok = th_v[rc_v[:-1] >= target]
    cands[f"Hit recall >= {target:.0%} (validation)"] = float(ok.max())
trows = []
for name, thr in cands.items():
    mv, mt = clf_metrics(y_val, p_val, thr), clf_metrics(y_test, p_te, thr)
    trows.append({"rule (chosen on validation)": name, "threshold": thr, "val precision": mv["precision"], "val hit recall": mv["hit_recall"], "val flop recall": mv["flop_recall"],
                  "TEST precision": mt["precision"], "TEST hit recall": mt["hit_recall"], "TEST flop recall": mt["flop_recall"], "TEST F1": mt["f1"], "TEST accuracy": mt["accuracy"],
                  "TEST Hits found": f"{mt['tp']} of {mt['tp'] + mt['fn']}", "TEST false positives": mt["fp"]})
thr_table = pd.DataFrame(trows)
display(thr_table.round(3))
results["threshold_table"] = thr_table.round(4).to_dict("records")
lo_r, hi_r = thr_table.iloc[0], thr_table.iloc[-2]
print(f"Reading the table: moving from the default threshold to '{hi_r['rule (chosen on validation)']}' raises test Hit recall from {lo_r['TEST hit recall']:.2f} to {hi_r['TEST hit recall']:.2f}, "
      f"but false positives go from {int(lo_r['TEST false positives'])} to {int(hi_r['TEST false positives'])} and Flop recall falls from {lo_r['TEST flop recall']:.2f} to {hi_r['TEST flop recall']:.2f}. "
      "Whether that trade is acceptable depends on the cost of missing a Hit versus backing a Flop; the data cannot decide that.")

fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
f_, t_, _ = roc_curve(y_val, p_val)
ax[0].plot(f_, t_, color="#4C72B0", lw=2, label=f"validation AUC {auc_fast(y_val, p_val):.2f}")
f2, t2, _ = roc_curve(y_test, p_te)
ax[0].plot(f2, t2, color="#C44E52", lw=2, label=f"test AUC {auc_fast(y_test, p_te):.2f}")
ax[0].plot([0, 1], [0, 1], "k--")
ax[0].set(title=f"ROC: {CH_MODEL} ({CH_GROUP.split(':')[0]})", xlabel="False positive rate", ylabel="Hit recall")
ax[0].legend()
ax[1].plot(rc_v, pr_v, color="#4C72B0", lw=2, label="validation")
pr_t, rc_t, _ = precision_recall_curve(y_test, p_te)
ax[1].plot(rc_t, pr_t, color="#C44E52", lw=2, label="test")
ax[1].axhline(y_val.mean(), color="#4C72B0", ls=":", lw=1)
ax[1].axhline(y_test.mean(), color="#C44E52", ls=":", lw=1)
ax[1].set(title="Precision-recall (dotted lines = base rates)", xlabel="Hit recall", ylabel="Precision", ylim=(0, 1.02))
ax[1].legend()
plt.savefig(OUT / "07_validation_vs_test_curves.png", bbox_inches="tight")
plt.show()

## 15. One pre-declared confirmation of the buzz question on the test period
The decision about buzz was made on validation data in section 11. As a **confirmation (not a selection)**, we now repeat the paired comparison once for the chosen model type on the test movies.
We do not use this to pick a feature group. Note that the test period has no YouTube records for 2026 releases and no usable data before 2017, so the YouTube features are only partly observable there.

In [ ]:
conf_rows = []
p_base_t = final_fit(CH_MODEL, base_g).predict_proba(test)[:, 1]
for g in GROUPS:
    pg = final_fit(CH_MODEL, g).predict_proba(test)[:, 1]
    if g == base_g:
        conf_rows.append({"feature group": g, "test AUC": auc_fast(y_test, pg), "gain vs base": np.nan, "ci_low": np.nan, "ci_high": np.nan})
    else:
        d, lo, hi, _ = paired_boot(y_test, pg, p_base_t)
        conf_rows.append({"feature group": g, "test AUC": auc_fast(y_test, pg), "gain vs base": d, "ci_low": lo, "ci_high": hi})
conf = pd.DataFrame(conf_rows)
display(conf.round(3))
results["confirmation_buzz_test"] = {"model": CH_MODEL, "table": conf.round(4).replace({np.nan: None}).to_dict("records")}
for _, r in conf.iloc[1:].iterrows():
    verdict = "interval excludes 0" if (r.ci_low > 0 or r.ci_high < 0) else "interval contains 0 (inconclusive)"
    print(f"  {r['feature group']}: gain {r['gain vs base']:+.3f} [{r.ci_low:+.3f}, {r.ci_high:+.3f}] -> {verdict}")

## 16. Feature importance (permutation importance on validation data)
**Permutation importance** shuffles one feature (or a group of features together) among the validation movies and measures how much the ROC-AUC drops. A large drop means the model relied on that information.
We compute it on the **out-of-fold validation movies** using the models that were trained on earlier periods, and average over the 5 folds (10 shuffles each). The test set is not touched.

**Reading it correctly.** Importance measures what the *model* uses, not what *causes* success. Correlated features share credit, so a feature can look unimportant when another one carries the same information.
SHAP values for the strongest boosted model are shown separately in section 17e; the grouped permutation analysis below remains the main importance measure.

In [ ]:
def importance_units(group):
    cols = ALL_GROUPS[group]
    w, yv = [c for c in cols if c in WIKI], [c for c in cols if c in YT]
    single = {"log_budget": ["log_budget"], "runtime": ["runtime"], "year": ["year"], "release month": ["month_sin", "month_cos"], "season": ["season"], "language": ["language"], "genres": ["genres"]}
    single.update({c: [c] for c in w + yv})
    grouped = {"Budget": ["log_budget"], "Runtime": ["runtime"], "Release timing (year, month, season)": ["year", "month_sin", "month_cos", "season"], "Language": ["language"], "Genre": ["genres"]}
    if w:
        grouped["Wikipedia buzz (all windows + buzz_per_budget)"] = w
    if yv:
        grouped["YouTube features"] = yv
    return single, grouped


def perm_importance(model_key, group, n_rep=10, seed=SEED):
    single, grouped = importance_units(group)
    rg = np.random.default_rng(seed)
    recs = []
    for k, (est, va) in fold_models[(model_key, group)].items():
        Xv, yv_ = dev.iloc[va].reset_index(drop=True), y_dev[va]
        base = auc_fast(yv_, est.predict_proba(Xv)[:, 1])
        for kind, units in (("feature", single), ("group", grouped)):
            for name, cols in units.items():
                for _ in range(n_rep):
                    Xp = Xv.copy()
                    Xp[cols] = Xv[cols].iloc[rg.permutation(len(Xv))].values
                    recs.append({"kind": kind, "unit": name, "fold": k, "auc_drop": base - auc_fast(yv_, est.predict_proba(Xp)[:, 1])})
    r = pd.DataFrame(recs)
    return r.groupby(["kind", "unit"]).auc_drop.agg(["mean", "std"]).reset_index().rename(columns={"mean": "auc_drop", "std": "sd"})


imp_targets = [(CH_MODEL, CH_GROUP)] + ([(CH_MODEL, list(GROUPS)[-1])] if CH_GROUP != list(GROUPS)[-1] else [])
imp_all = {}
for mk, g in imp_targets:
    imp = perm_importance(mk, g)
    imp_all[(mk, g)] = imp
    feat = imp[imp.kind == "feature"].sort_values("auc_drop", ascending=False).head(10)
    grp = imp[imp.kind == "group"].sort_values("auc_drop", ascending=False)
    print(f"\n=== {mk} on '{g}' (validation folds) ===")
    print("Top 10 features by AUC drop when shuffled:")
    display(feat[["unit", "auc_drop", "sd"]].round(4).reset_index(drop=True))
    print("Grouped importance (all columns of a source shuffled together):")
    display(grp[["unit", "auc_drop", "sd"]].round(4).reset_index(drop=True))
    fig, ax = plt.subplots(1, 2, figsize=(14, 4.2))
    ax[0].barh(feat.unit[::-1], feat.auc_drop[::-1], xerr=feat.sd[::-1], color="#55A868")
    ax[0].set(title="Top 10 features", xlabel="drop in validation AUC when shuffled")
    ax[1].barh(grp.unit[::-1], grp.auc_drop[::-1], xerr=grp.sd[::-1], color="#4C72B0")
    ax[1].set(title="Information sources", xlabel="drop in validation AUC when shuffled")
    plt.suptitle(f"{mk} | {g}", y=1.02)
    plt.savefig(OUT / f"08_importance_{mk.split()[0].lower()}_{g.split(':')[0].lower()}.png", bbox_inches="tight")
    plt.show()
    results.setdefault("permutation_importance", {})[f"{mk} | {g}"] = {"top_features": feat.round(5).to_dict("records"), "groups": grp.round(5).to_dict("records")}

# ---- how much does buzz_per_budget repeat log_budget? ----
ov = dev.dropna(subset=["pv_30"])
Xo = np.column_stack([np.ones(len(ov)), ov.log_budget, ov.l_pv_90_31, ov.l_pv_30_8, ov.l_pv_7_1])
coef, *_ = np.linalg.lstsq(Xo, ov.buzz_per_budget.values, rcond=None)
r2_bpb = 1 - ((ov.buzz_per_budget.values - Xo @ coef) ** 2).sum() / ((ov.buzz_per_budget.values - ov.buzz_per_budget.mean()) ** 2).sum()
print(f"\nOverlap check: buzz_per_budget = log10(1 + pv_30) - log_budget. A linear fit on log_budget and the three page-view windows reproduces {r2_bpb:.0%} of its variance, "
      f"so it adds almost no new information for a linear model. Its correlation with log_budget alone is {ov[['buzz_per_budget', 'log_budget']].corr().iloc[0, 1]:.2f}, "
      "but it contains the budget inside it, so when it ranks as important, part of that credit belongs to budget (and to page views). The ablation in section 17 removes it to check.")
results["buzz_per_budget_overlap_r2"] = float(r2_bpb)

## 17. Robustness and sanity checks
All checks below use the **development set with chronological out-of-fold predictions**, never the test set, and reuse the hyperparameters already chosen for the chosen model type (no new tuning on the test period).
Each check ends with what it **can** and **cannot** show.

| Check | What it can show | What it cannot show |
|---|---|---|
| Shuffled labels | The pipeline does not produce good scores from noise | That leakage is absent (a leak that survives shuffling, or one inside the true labels, would not be caught) |
| Other ROI thresholds | The conclusions do not depend on the exact "ROI >= 2" definition | That any threshold is the right business definition of a Hit |
| Missing buzz | Whether results depend on which movies have buzz data | Why the data is missing (the YouTube sample was not random) |
| Language and year splits | Where the model is more or less accurate | Real group differences when the groups are small (intervals are wide) |
| Data source and outlier checks | Whether suspicious budget/revenue values drive the result | Which budgets are actually correct |
| Dummy baseline | The model beats guessing | That the model is useful for a real decision |

In [ ]:
def params_for(group):
    return final_fit(CH_MODEL, group if group in GROUPS else list(GROUPS)[1]).best_params_


def oof_groups(df, y, groups, n_splits=5):
    """Out-of-fold predictions for several feature groups on any (sub)set of development movies, with fixed hyperparameters."""
    folds = list(TimeSeriesSplit(n_splits=n_splits).split(df))
    params = {g: params_for(g) for g in groups}            # looked up once here, so workers never re-tune

    def one(g, tr, va):
        return g, va, make_pipe(CH_MODEL, g, params[g]).fit(df.iloc[tr], y[tr]).predict_proba(df.iloc[va])[:, 1]

    outp = Parallel(n_jobs=-1)(delayed(one)(g, tr, va) for g in groups for tr, va in folds)
    preds = {g: np.full(len(df), np.nan) for g in groups}
    for g, va, p in outp:
        preds[g][va] = p
    idx = np.concatenate([va for _, va in folds])
    return preds, idx


def gains_table(df, y, label, groups=None):
    groups = groups or list(GROUPS)
    preds, idx = oof_groups(df, y, groups)
    row = {"setting": label, "movies": len(df), "hit rate": y.mean()}
    for g in groups:
        row[g.split(": ")[0] + " AUC"] = auc_fast(y[idx], preds[g][idx])
    if len(groups) == 4:
        d, lo, hi, _ = paired_boot(y[idx], preds[groups[3]][idx], preds[groups[0]][idx])
        row["D minus A"] = f"{d:+.3f} [{lo:+.3f}, {hi:+.3f}]"
    return row, preds, idx


# ---- 1. shuffled labels ----
real_preds, real_idx = oof_groups(dev, y_dev, [CH_GROUP])
real_auc = auc_fast(y_dev[real_idx], real_preds[CH_GROUP][real_idx])


P_CH = params_for(CH_GROUP)


def shuffled_auc(seed):
    yp = np.random.default_rng(seed).permutation(y_dev)
    folds = list(TimeSeriesSplit(n_splits=N_OUTER).split(dev))
    p = np.full(len(dev), np.nan)
    for tr, va in folds:
        p[va] = make_pipe(CH_MODEL, CH_GROUP, P_CH).fit(dev.iloc[tr], yp[tr]).predict_proba(dev.iloc[va])[:, 1]
    idx = np.concatenate([va for _, va in folds])
    return auc_fast(yp[idx], p[idx])


shuf = np.array(Parallel(n_jobs=-1)(delayed(shuffled_auc)(s) for s in range(30)))
p_shuf = (1 + (shuf >= real_auc).sum()) / (len(shuf) + 1)
print(f"1. Shuffled labels ({CH_MODEL}, {CH_GROUP}): real validation AUC {real_auc:.3f}; 30 shuffled-label runs give mean {shuf.mean():.3f} (min {shuf.min():.3f}, max {shuf.max():.3f}); "
      f"empirical p = {p_shuf:.3f}.")
print("   Establishes: the scores are not produced by chance. Does not establish: absence of leakage.")
results["shuffled_labels"] = {"real_auc": float(real_auc), "shuffled_mean": float(shuf.mean()), "shuffled_min": float(shuf.min()), "shuffled_max": float(shuf.max()), "p": float(p_shuf)}
plt.figure(figsize=(6.5, 3.6))
plt.hist(shuf, bins=12, color="#8172B2", label="shuffled labels")
plt.axvline(real_auc, color="crimson", lw=2, label=f"real labels ({real_auc:.2f})")
plt.axvline(0.5, color="k", ls="--", lw=1)
plt.title("Shuffled-label sanity check (validation AUC)")
plt.legend()
plt.savefig(OUT / "09_shuffled_labels.png", bbox_inches="tight")
plt.show()

# ---- 2. other Hit definitions ----
roi_rows = []
for thr in (1.0, 1.5, 2.0, 3.0):
    row, _, _ = gains_table(dev, (dev.roi >= thr).astype(int).values, f"Hit if ROI >= {thr}")
    row["test hit rate (descriptive)"] = float((test.roi >= thr).mean())
    roi_rows.append(row)
roi_tab = pd.DataFrame(roi_rows)
print("\n2. Different Hit definitions (validation AUC by feature group; the dummy baseline is 0.50 everywhere):")
display(roi_tab.round(3))
results["robust_roi_thresholds"] = roi_tab.round(4).to_dict("records")

# ---- 3. missing buzz ----
p_ch = oof[(CH_MODEL, CH_GROUP)]
has_w = dev.pv_30.notna().values[val_idx]
has_y = (dev.in_yt_file.values == 1)[val_idx]
sub_rows = []
for name, mask in [("has Wikipedia page views", has_w), ("no Wikipedia page views", ~has_w), ("has YouTube record", has_y), ("no YouTube record", ~has_y)]:
    yy, pp = y_dev[val_idx][mask], p_ch[val_idx][mask]
    ok = yy.sum() >= 10 and (len(yy) - yy.sum()) >= 10
    lo, hi = boot_auc_ci(yy, pp) if ok else (np.nan, np.nan)
    sub_rows.append({"validation movies": name, "n": int(mask.sum()), "hit rate": yy.mean(), "AUC (chosen model)": auc_fast(yy, pp) if ok else np.nan, "ci_low": lo, "ci_high": hi})
print(f"\n3a. Chosen model's validation AUC by availability of buzz data ({CH_MODEL}, {CH_GROUP}):")
display(pd.DataFrame(sub_rows).round(3))
cc_rows = []
for label, mask, gs_ in [("Wikipedia data present", dev.pv_30.notna().values, [base_g, list(GROUPS)[1]]),
                         ("YouTube record present", (dev.in_yt_file.values == 1), [base_g, list(GROUPS)[2]]),
                         ("both present", dev.pv_30.notna().values & (dev.in_yt_file.values == 1), [base_g, list(GROUPS)[3]])]:
    sub = dev[mask].reset_index(drop=True)
    ys = sub.hit.values
    pr_, ix = oof_groups(sub, ys, gs_)
    d, lo, hi, pv = paired_boot(ys[ix], pr_[gs_[1]][ix], pr_[gs_[0]][ix])
    cc_rows.append({"complete-case subset": label, "movies": len(sub), "AUC base": auc_fast(ys[ix], pr_[gs_[0]][ix]), "AUC with buzz": auc_fast(ys[ix], pr_[gs_[1]][ix]),
                    "gain": d, "ci_low": lo, "ci_high": hi, "p_bootstrap": pv})
cc_tab = pd.DataFrame(cc_rows)
cc_tab["p_adjusted_BH"] = bh_adjust(cc_tab.p_bootstrap.values)
print("3b. Complete-case check: only movies where the buzz data really exists (no imputation needed). These are post-hoc subgroup analyses, so adjusted p-values are shown:")
display(cc_tab.round(3))
results["robust_missing_buzz"] = {"by_availability": pd.DataFrame(sub_rows).round(4).replace({np.nan: None}).to_dict("records"), "complete_case": cc_tab.round(4).to_dict("records")}

# ---- 4. language-wise and 5. year-wise performance ----
def split_table(df_part, p_part, y_part, col, min_n=40, min_class=10):
    rows = []
    for key, idx in df_part.groupby(col).indices.items():
        yy, pp = y_part[idx], p_part[idx]
        ok = len(yy) >= min_n and yy.sum() >= min_class and (len(yy) - yy.sum()) >= min_class
        lo, hi = boot_auc_ci(yy, pp) if ok else (np.nan, np.nan)
        rows.append({col: key, "n": len(yy), "hit rate": yy.mean(), "AUC": auc_fast(yy, pp) if ok else np.nan, "ci_low": lo, "ci_high": hi, "note": "" if ok else "too few movies or one class too small"})
    return pd.DataFrame(rows)


dev_val = dev.iloc[val_idx].reset_index(drop=True)
lang_val = split_table(dev_val, p_ch[val_idx], y_dev[val_idx], "language")
lang_test = split_table(test, p_te, y_test, "language")
print(f"4. Language-wise AUC of the chosen model ({CH_MODEL}, {CH_GROUP}). Validation (out-of-fold):")
display(lang_val.round(3))
print("   Test period (frozen model, descriptive only):")
display(lang_test.round(3))
lang_shift = pd.DataFrame({"dev hit rate": dev.groupby("language").hit.mean(), "test hit rate": test.groupby("language").hit.mean()})
lang_shift["change"] = lang_shift["test hit rate"] - lang_shift["dev hit rate"]
print("   Hit rate by language, development versus test period (the label shift is not the same in every language):")
display(lang_shift.round(3))
dev_val["year"] = dev_val.release_date.dt.year
year_val = split_table(dev_val, p_ch[val_idx], y_dev[val_idx], "year", min_n=40, min_class=10)
year_test = split_table(test, p_te, y_test, "year", min_n=40, min_class=10)
print("5. Year-wise AUC. Validation (out-of-fold):")
display(year_val.round(3))
print("   Test period (frozen model, descriptive only):")
display(year_test.round(3))
results["robust_language"] = {"validation": lang_val.round(4).replace({np.nan: None}).to_dict("records"), "test": lang_test.round(4).replace({np.nan: None}).to_dict("records")}
results["robust_year"] = {"validation": year_val.round(4).replace({np.nan: None}).to_dict("records"), "test": year_test.round(4).replace({np.nan: None}).to_dict("records")}

# ---- 6. data-quality sensitivity and the buzz_per_budget ablation ----
sens = []
r0, _, _ = gains_table(dev, y_dev, "all development movies")
sens.append(r0)
sub = dev[dev.tmdb_only == 1].reset_index(drop=True)
sens.append(gains_table(sub, sub.hit.values, "only tmdb/tmdb budget and revenue")[0])
sub = dev[dev.suspect_roi == 0].reset_index(drop=True)
sens.append(gains_table(sub, sub.hit.values, "without extreme-ROI movies (ROI < 0.05 or > 50)")[0])
sens.append(gains_table(dev.assign(year=0), y_dev, "release-year feature removed (set constant)")[0])
sens_tab = pd.DataFrame(sens)
print("6. Data-quality sensitivity (validation AUC by feature group; same chronological folds):")
display(sens_tab.round(3))
ab_preds, ab_idx = oof_groups(dev, y_dev, [base_g, list(GROUPS)[1], "B-: base + Wikipedia without buzz_per_budget"])
ab = {g: auc_fast(y_dev[ab_idx], ab_preds[g][ab_idx]) for g in ab_preds}
d_ab, lo_ab, hi_ab, _ = paired_boot(y_dev[ab_idx], ab_preds["B-: base + Wikipedia without buzz_per_budget"][ab_idx], ab_preds[base_g][ab_idx])
print("   buzz_per_budget ablation (validation AUC): " + "; ".join(f"{g.split(':')[0]}: {v:.3f}" for g, v in ab.items())
      + f". Wikipedia buzz without the budget-containing ratio vs base: {d_ab:+.3f} [{lo_ab:+.3f}, {hi_ab:+.3f}].")
results["robust_sensitivity"] = {"table": sens_tab.round(4).to_dict("records"), "ablation_auc": {k: float(v) for k, v in ab.items()}, "ablation_gain_vs_base": [float(d_ab), float(lo_ab), float(hi_ab)]}
print(f"7. Baseline: the DummyClassifier has AUC 0.50 in every table above; the chosen model's validation AUC is {real_auc:.3f} and its test AUC is {auc_fast(y_test, p_te):.3f}.")

## 17b. Deep learning models (MLP, GRU, 1D-CNN)
This is a course on machine learning and data science, so we also test neural networks, built with PyTorch and evaluated **exactly like the classic models** (same chronological folds, same validation movies, same paired bootstrap).

| Network | What it reads | Why it fits this data |
|---|---|---|
| **MLP** (multi-layer perceptron) | all pre-release features at once | the standard neural network for tabular data |
| **GRU** (a recurrent network, RNN) | the three Wikipedia page-view windows as a **time sequence** (90-31 days, 30-8 days, 7-1 days before release), plus the other features | page views before release really are a short time series |
| **1D-CNN** (convolutional network) | the same 3-step sequence, scanned by a small convolution filter | tests whether local changes in attention (for example a rising trend) carry signal |

**What this can and cannot show.** There are only 578 development movies and the "sequence" has just 3 steps, so deep networks have little to learn from. We use small networks (16 units, dropout, weight decay, early stopping on the last 20% of each training window, average of 3 random seeds) with **fixed settings and no grid search**. We do not expect them to beat the classic models, and a fair test is still worth reporting. A CNN is normally used on images (posters, trailer frames) or text (plot summaries); those inputs are not in our two files, so that is future work.

Selection rule: as before, nothing here uses the test set. For each network type the feature group with the best mean validation AUC is picked, and only that one is scored once on the test period as a supplementary result.

In [ ]:
import torch
import torch.nn as nn
torch.set_num_threads(1)
SEQ_NAMES = ("l_pv_90_31", "l_pv_30_8", "l_pv_7_1")


class _Net(nn.Module):
    def __init__(self, kind, n_static, hidden, dropout):
        super().__init__()
        self.kind = kind
        if kind == "GRU":
            self.seq, extra = nn.GRU(1, hidden, batch_first=True), hidden
        elif kind == "1D-CNN":
            self.seq, extra = nn.Conv1d(1, hidden, kernel_size=2), hidden
        else:
            self.seq, extra = None, 0
        self.head = nn.Sequential(nn.Linear(n_static + extra, 2 * hidden), nn.ReLU(), nn.Dropout(dropout), nn.Linear(2 * hidden, 1))

    def forward(self, xs, xq):
        if self.kind == "GRU":
            z = torch.cat([xs, self.seq(xq)[1][-1]], 1)
        elif self.kind == "1D-CNN":
            z = torch.cat([xs, torch.relu(self.seq(xq.transpose(1, 2))).amax(2)], 1)
        else:
            z = xs
        return self.head(z).squeeze(1)


class TorchNet(BaseEstimator, ClassifierMixin):
    """Small neural network that sits inside the scikit-learn Pipeline (so scaling and imputation are fitted on training rows only)."""

    def __init__(self, kind="MLP", hidden=16, dropout=0.3, lr=3e-3, weight_decay=1e-2, max_epochs=120, patience=12, n_seeds=3):
        self.kind, self.hidden, self.dropout, self.lr, self.weight_decay = kind, hidden, dropout, lr, weight_decay
        self.max_epochs, self.patience, self.n_seeds = max_epochs, patience, n_seeds

    def _tensors(self, X):
        cols = list(X.columns)
        if self.kind == "MLP":
            seq_idx = []
        else:
            seq_idx = [next(i for i, c in enumerate(cols) if c.endswith("__" + n)) for n in SEQ_NAMES]
        static_idx = [i for i in range(len(cols)) if i not in seq_idx]
        A = np.asarray(X, dtype=np.float32)
        xs = torch.tensor(A[:, static_idx])
        xq = torch.tensor(A[:, seq_idx]).unsqueeze(2) if seq_idx else torch.zeros(len(A), 1, 1)
        return xs, xq, len(static_idx)

    def fit(self, X, y):
        self.classes_ = np.array([0, 1])
        xs, xq, n_static = self._tensors(X)
        yt = torch.tensor(np.asarray(y), dtype=torch.float32)
        n = len(yt)
        n_val = max(int(0.2 * n), 20)
        tr, va = np.arange(0, n - n_val), np.arange(n - n_val, n)       # rows arrive in release-date order, so the last 20% is the most recent
        loss_fn = nn.BCEWithLogitsLoss()
        self.nets_ = []
        for seed in range(self.n_seeds):
            torch.manual_seed(SEED + seed)
            net = _Net(self.kind, n_static, self.hidden, self.dropout)
            opt = torch.optim.AdamW(net.parameters(), lr=self.lr, weight_decay=self.weight_decay)
            g = torch.Generator().manual_seed(SEED + seed)
            best, best_state, wait = np.inf, None, 0
            for _ in range(self.max_epochs):
                net.train()
                perm = torch.tensor(tr)[torch.randperm(len(tr), generator=g)]
                for b in range(0, len(perm), 64):
                    idx = perm[b:b + 64]
                    opt.zero_grad()
                    loss_fn(net(xs[idx], xq[idx]), yt[idx]).backward()
                    opt.step()
                net.eval()
                with torch.no_grad():
                    v = loss_fn(net(xs[va], xq[va]), yt[va]).item()
                if v < best - 1e-4:
                    best, wait = v, 0
                    best_state = {k: t.clone() for k, t in net.state_dict().items()}
                else:
                    wait += 1
                    if wait >= self.patience:
                        break
            net.load_state_dict(best_state)
            net.eval()
            self.nets_.append(net)
        return self

    def predict_proba(self, X):
        xs, xq, _ = self._tensors(X)
        with torch.no_grad():
            p = np.mean([torch.sigmoid(net(xs, xq)).numpy() for net in self.nets_], axis=0)
        return np.column_stack([1 - p, p])

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)


def make_dl_pipe(kind, group):
    prep = make_preprocessor(group, True).set_output(transform="pandas")
    return Pipeline([("prep", prep), ("model", TorchNet(kind=kind))])


G_B, G_D = list(GROUPS)[1], list(GROUPS)[3]
DL_COMBOS = [("MLP", g) for g in GROUPS] + [("GRU", G_B), ("GRU", G_D), ("1D-CNN", G_B), ("1D-CNN", G_D)]


def dl_fold(kind, group, k, tr, va):
    pipe = make_dl_pipe(kind, group).fit(dev.iloc[tr], y_dev[tr])
    return kind, group, k, va, pipe.predict_proba(dev.iloc[va])[:, 1]


t_dl = time.time()
out = Parallel(n_jobs=-1)(delayed(dl_fold)(kd, g, k, tr, va) for kd, g in DL_COMBOS for k, (tr, va) in enumerate(outer_folds))
dl_oof = {}
for kind, group, k, va, p in out:
    dl_oof.setdefault((kind, group), np.full(len(dev), np.nan))[va] = p
print(f"Deep-learning validation finished in {time.time() - t_dl:.0f} s ({len(out)} fits, 3 seeds each).")

rows = []
for (kind, group), p_all in dl_oof.items():
    fa = [auc_fast(y_dev[va], p_all[va]) for _, va in outer_folds]
    lo, hi = boot_auc_ci(y_dev[val_idx], p_all[val_idx])
    cls = real[real["feature group"] == group].sort_values("cv_auc_mean", ascending=False).iloc[0]
    d, dlo, dhi, p = paired_boot(y_dev[val_idx], p_all[val_idx], oof[(cls.model, group)][val_idx])
    rows.append({"network": kind, "feature group": group, "cv_auc_mean": np.mean(fa), "cv_auc_sd_over_folds": np.std(fa, ddof=1), "pooled_oof_auc": auc_fast(y_dev[val_idx], p_all[val_idx]),
                 "ci_low": lo, "ci_high": hi, "best classic model, same features": cls.model, "classic cv_auc_mean": cls.cv_auc_mean,
                 "gain vs that classic model": d, "gain_ci_low": dlo, "gain_ci_high": dhi, "p_bootstrap": p})
dl_val = pd.DataFrame(rows)
dl_val["p_adjusted_BH"] = bh_adjust(dl_val.p_bootstrap.values)
dl_val["better_convincingly"] = (dl_val.gain_ci_low > 0) & (dl_val.p_adjusted_BH < 0.05)
print("Validation results of the neural networks (same 5 chronological folds). 'gain' is the paired AUC difference to the best classic model with the same features:")
display(dl_val.round(3))
results["deep_learning_validation"] = dl_val.round(4).to_dict("records")

In [ ]:
# One scoring on the test period for the best feature group of each network type (chosen by validation AUC only).
dl_best = dl_val.loc[dl_val.groupby("network").cv_auc_mean.idxmax()].set_index("network")
dl_rows = []
for kind in ["MLP", "GRU", "1D-CNN"]:
    group = dl_best.loc[kind, "feature group"]
    pipe = make_dl_pipe(kind, group).fit(dev, y_dev)
    p = pipe.predict_proba(test)[:, 1]
    lo, hi = boot_auc_ci(y_test, p)
    dl_rows.append({"model": kind + " (neural network)", "feature group": group, "validation cv_auc_mean": dl_best.loc[kind, "cv_auc_mean"], "test AUC": auc_fast(y_test, p), "ci_low": lo, "ci_high": hi, "test Brier": brier_score_loss(y_test, p)})
for mk in MODELS:
    v = results["final_test"][mk]
    dl_rows.append({"model": mk + " (classic)", "feature group": v["group"], "validation cv_auc_mean": best_per_model.loc[mk, "cv_auc_mean"], "test AUC": v["auc"], "ci_low": v["ci"]["auc"][0], "ci_high": v["ci"]["auc"][1], "test Brier": v["brier"]})
dl_test = pd.DataFrame(dl_rows)
print("Supplementary test results (one scoring each; the classic rows are repeated from section 13 for comparison). These were not used to choose anything:")
display(dl_test.round(3))
results["deep_learning_test"] = dl_test.round(4).to_dict("records")

fig, ax = plt.subplots(figsize=(9, 4.6))
plot = pd.concat([dl_val.assign(name=lambda d: d.network + " | " + d["feature group"].str.split(": ").str[1])[["name", "cv_auc_mean", "cv_auc_sd_over_folds"]],
                  real.assign(name=lambda d: d.model + " | " + d["feature group"].str.split(": ").str[1])[["name", "cv_auc_mean", "cv_auc_sd_over_folds"]]]).sort_values("cv_auc_mean")
colors = ["#C44E52" if n.split(" | ")[0] in ("MLP", "GRU", "1D-CNN") else "#4C72B0" for n in plot.name]
ax.barh(plot.name, plot.cv_auc_mean, xerr=plot.cv_auc_sd_over_folds, color=colors)
ax.axvline(0.5, color="k", ls="--", lw=1)
ax.set(xlim=(0.4, 0.85), xlabel="mean validation AUC over 5 chronological folds (bars = sd over folds)", title="Neural networks (red) versus classic models (blue)")
ax.tick_params(axis="y", labelsize=7)
plt.savefig(OUT / "08b_deep_learning_vs_classic.png", bbox_inches="tight")
plt.show()
best_dl = dl_val.loc[dl_val.cv_auc_mean.idxmax()]
n_dl_win = int(dl_val.better_convincingly.sum())
print(f"Best neural network on validation: {best_dl.network} on '{best_dl['feature group'].split(': ')[1]}' with mean AUC {best_dl.cv_auc_mean:.3f}, "
      f"against {chosen_row.cv_auc_mean:.3f} for the chosen classic model. Neural networks that beat the best classic model with the same features convincingly: {n_dl_win} of {len(dl_val)}.")

## 17c. More models: a wider comparison
To check that the result is not just "bad luck with three models", we train **more model families** with the same procedure (same chronological folds, same inner tuning, same four feature groups):

| Model | Idea |
|---|---|
| Extra Trees | like a random forest, but with random split points (less variance) |
| AdaBoost | many tiny trees, each focusing on the previous mistakes |
| XGBoost, LightGBM | popular, fast gradient-boosting libraries |
| SVM (RBF kernel) | finds a smooth curved boundary between Hits and Flops |
| k-Nearest Neighbours | a movie is predicted like the most similar movies |
| Gaussian Naive Bayes | a simple probabilistic model that treats features as independent |
| CatBoost | a gradient-boosting library with symmetric trees, often steady on small data |
| Elastic-net Logistic Regression | a linear model with a mix of L1 and L2 penalties; it can switch useless features off |
| Gradient Boosting (scikit-learn) | the classic gradient-boosted trees, with shallow trees and a small learning rate |
| Linear Discriminant Analysis | a classic linear classifier that models each class as a bell-shaped cloud |
| Average ensemble | the mean probability of Logistic Regression, Random Forest and HistGradientBoosting |
| Stacking | a small Logistic Regression learns how much to trust each of the three main models; it is trained only on earlier, honest out-of-fold predictions (for the first validation fold, which has none, it falls back to the simple average) |

**A warning about trying many models.** The more models we compare, the more likely the "best" one is best only by luck. The best validation AUC from a long list is therefore **optimistic**, and differences of about 0.01 are not meaningful. As before, the test set is not used to choose anything: each model type is scored once, with the feature group that was best on validation, and the results are supplementary.

In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import AdaBoostClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

EXTRA = {
    "Extra Trees": (lambda: ExtraTreesClassifier(n_estimators=300, random_state=SEED, n_jobs=1), {"model__min_samples_leaf": [3, 8, 15], "model__max_features": ["sqrt", 0.5]}, False),
    "AdaBoost": (lambda: AdaBoostClassifier(n_estimators=150, random_state=SEED), {"model__learning_rate": [0.05, 0.2, 0.5]}, False),
    "SVM (RBF)": (lambda: SVC(kernel="rbf", probability=True, random_state=SEED), {"model__C": [0.3, 1.0, 3.0]}, True),
    "k-Nearest Neighbours": (lambda: KNeighborsClassifier(), {"model__n_neighbors": [5, 10, 20]}, True),
    "Gaussian Naive Bayes": (lambda: GaussianNB(), {"model__var_smoothing": [1e-9, 1e-6, 1e-3]}, True),
    "Elastic-net logistic regression": (lambda: LogisticRegression(penalty="elasticnet", solver="saga", l1_ratio=0.5, max_iter=5000, random_state=SEED), {"model__C": [0.03, 0.1, 0.3, 1.0]}, True),
    "Gradient Boosting (sklearn)": (lambda: GradientBoostingClassifier(n_estimators=150, subsample=0.8, random_state=SEED), {"model__max_depth": [2, 3], "model__learning_rate": [0.03, 0.08], "model__min_samples_leaf": [10, 25]}, False),
    "Linear discriminant analysis": (lambda: LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto"), {"model__tol": [1e-4]}, True),
}
try:
    from xgboost import XGBClassifier
    EXTRA["XGBoost"] = (lambda: XGBClassifier(n_estimators=150, subsample=0.8, tree_method="hist", eval_metric="logloss", n_jobs=1, random_state=SEED, verbosity=0),
                        {"model__max_depth": [2, 3], "model__learning_rate": [0.03, 0.08], "model__min_child_weight": [1, 5]}, False)
except ImportError:
    print("xgboost is not installed here; it is skipped.")
try:
    from lightgbm import LGBMClassifier
    EXTRA["LightGBM"] = (lambda: LGBMClassifier(n_estimators=150, subsample=0.8, subsample_freq=1, random_state=SEED, n_jobs=1, verbose=-1),
                         {"model__num_leaves": [4, 8], "model__learning_rate": [0.03, 0.08], "model__min_child_samples": [15, 30]}, False)
except ImportError:
    print("lightgbm is not installed here; it is skipped.")

try:
    try:
        from catboost import CatBoostClassifier
    except ImportError:                                           # Colab does not ship CatBoost, so install it once
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "catboost"])
        from catboost import CatBoostClassifier
    EXTRA["CatBoost"] = (lambda: CatBoostClassifier(iterations=150, random_seed=SEED, verbose=0, thread_count=1, allow_writing_files=False),
                         {"model__depth": [2, 4], "model__learning_rate": [0.03, 0.08], "model__l2_leaf_reg": [3, 10]}, False)
except Exception as err:                                          # network or build problem: carry on without it, and say so
    print(f"CatBoost is not available here and is skipped: {type(err).__name__}: {err}")

STACK = "Stacking (LR on LR + RF + HGB)"


def tune_extra(X, y, model_key, group, n_inner=3):
    est, grid, scale = EXTRA[model_key]
    pipe = Pipeline([("prep", make_preprocessor(group, scale)), ("model", est())])
    return GridSearchCV(pipe, grid, scoring="roc_auc", cv=TimeSeriesSplit(n_splits=n_inner), n_jobs=1, refit=True).fit(X, y)


def extra_fold(model_key, group, k, tr, va):
    gs = tune_extra(dev.iloc[tr], y_dev[tr], model_key, group)
    return model_key, group, k, va, gs.predict_proba(dev.iloc[va])[:, 1]


t_ex = time.time()
out = Parallel(n_jobs=-1)(delayed(extra_fold)(m, g, k, tr, va) for m in EXTRA for g in GROUPS for k, (tr, va) in enumerate(outer_folds))
ex_oof = {}
for m, g, k, va, p in out:
    ex_oof.setdefault((m, g), np.full(len(dev), np.nan))[va] = p
for g in GROUPS:                                         # average ensemble of the three tuned classic models (all out-of-fold)
    ex_oof[("Average ensemble (LR + RF + HGB)", g)] = np.mean([oof[(m, g)] for m in MODELS], axis=0)
for g in GROUPS:                                         # stacking: the meta-learner for fold k learns only from out-of-fold predictions of folds before k
    base = np.column_stack([oof[(m, g)] for m in MODELS])
    st = base.mean(axis=1)                                # fold 1 has no earlier predictions to learn from, so it uses the plain average
    for k, (tr, va) in enumerate(outer_folds):
        if k == 0:
            continue
        prev = np.concatenate([outer_folds[j][1] for j in range(k)])
        meta = LogisticRegression(C=1.0, max_iter=1000).fit(base[prev], y_dev[prev])
        st[va] = meta.predict_proba(base[va])[:, 1]
    ex_oof[(STACK, g)] = st
print(f"Extra models validated in {time.time() - t_ex:.0f} s ({len(out)} outer fits, each with an inner grid search).")

rows = []
for (m, g), p_all in ex_oof.items():
    fa = [auc_fast(y_dev[va], p_all[va]) for _, va in outer_folds]
    lo, hi = boot_auc_ci(y_dev[val_idx], p_all[val_idx])
    cls = real[real["feature group"] == g].sort_values("cv_auc_mean", ascending=False).iloc[0]
    d, dlo, dhi, p = paired_boot(y_dev[val_idx], p_all[val_idx], oof[(cls.model, g)][val_idx])
    rows.append({"model": m, "feature group": g, "cv_auc_mean": np.mean(fa), "cv_auc_sd_over_folds": np.std(fa, ddof=1), "pooled_oof_auc": auc_fast(y_dev[val_idx], p_all[val_idx]),
                 "ci_low": lo, "ci_high": hi, "brier": brier_score_loss(y_dev[val_idx], np.clip(p_all[val_idx], 0, 1)),
                 "best of LR/RF/HGB, same features": cls.model, "gain vs that model": d, "gain_ci_low": dlo, "gain_ci_high": dhi, "p_bootstrap": p})
ex_val = pd.DataFrame(rows)
ex_val["p_adjusted_BH"] = bh_adjust(ex_val.p_bootstrap.values)
ex_val["better_convincingly"] = (ex_val.gain_ci_low > 0) & (ex_val.p_adjusted_BH < 0.05)
ex_val["worse_convincingly"] = (ex_val.gain_ci_high < 0) & (ex_val.p_adjusted_BH < 0.05)
print("Validation results of the additional models. 'gain' is the paired AUC difference to the best of Logistic Regression / Random Forest / HistGradientBoosting with the same features:")
display(ex_val.drop(columns=["worse_convincingly"]).round(3))
results["extra_models_validation"] = ex_val.round(4).to_dict("records")

In [ ]:
# One test scoring per model type, using the feature group that was best on validation only (supplementary, nothing is selected with it).
ex_best = ex_val.loc[ex_val.groupby("model").cv_auc_mean.idxmax()].set_index("model")
test_rows = []
for m in ex_best.index:
    g = ex_best.loc[m, "feature group"]
    if m.startswith("Average ensemble"):
        p = np.mean([final_fit(mk, g).predict_proba(test)[:, 1] for mk in MODELS], axis=0)
    elif m == STACK:                                      # meta-learner trained on all out-of-fold development predictions; base models refitted on all development data
        base_dev = np.column_stack([oof[(mk, g)][val_idx] for mk in MODELS])
        meta = LogisticRegression(C=1.0, max_iter=1000).fit(base_dev, y_dev[val_idx])
        p = meta.predict_proba(np.column_stack([final_fit(mk, g).predict_proba(test)[:, 1] for mk in MODELS]))[:, 1]
    else:
        p = tune_extra(dev, y_dev, m, g, n_inner=N_OUTER).predict_proba(test)[:, 1]
    lo, hi = boot_auc_ci(y_test, p)
    test_rows.append({"model": m, "feature group": g, "validation cv_auc_mean": ex_best.loc[m, "cv_auc_mean"], "test AUC": auc_fast(y_test, p), "ci_low": lo, "ci_high": hi, "test Brier": brier_score_loss(y_test, p)})
ex_test = pd.DataFrame(test_rows)
print("Supplementary test results of the additional models (one scoring each):")
display(ex_test.round(3))
results["extra_models_test"] = ex_test.round(4).to_dict("records")

# ---- one leaderboard for every model family ----
lb = []
for mk in MODELS:
    v = results["final_test"][mk]
    lb.append({"family": "classic (main)", "model": mk, "feature group": v["group"], "validation AUC": best_per_model.loc[mk, "cv_auc_mean"], "test AUC": v["auc"], "ci_low": v["ci"]["auc"][0], "ci_high": v["ci"]["auc"][1]})
for _, r in ex_test.iterrows():
    lb.append({"family": "classic (extra)", "model": r.model, "feature group": r["feature group"], "validation AUC": r["validation cv_auc_mean"], "test AUC": r["test AUC"], "ci_low": r.ci_low, "ci_high": r.ci_high})
for _, r in dl_test[dl_test.model.str.contains("neural")].iterrows():
    lb.append({"family": "neural network", "model": r.model.replace(" (neural network)", ""), "feature group": r["feature group"], "validation AUC": r["validation cv_auc_mean"], "test AUC": r["test AUC"], "ci_low": r.ci_low, "ci_high": r.ci_high})
leaderboard = pd.DataFrame(lb).sort_values("validation AUC", ascending=False).reset_index(drop=True)
print("LEADERBOARD of all model families (sorted by validation AUC; the test AUC is a single supplementary scoring):")
display(leaderboard.round(3))
results["leaderboard"] = leaderboard.round(4).to_dict("records")

fig, ax = plt.subplots(figsize=(9, 6))
lb_plot = leaderboard.iloc[::-1]
pos = np.arange(len(lb_plot))
cmap = {"classic (main)": "#4C72B0", "classic (extra)": "#55A868", "neural network": "#C44E52"}
ax.barh(pos, lb_plot["validation AUC"], color=[cmap[f] for f in lb_plot.family], alpha=0.85, label="validation AUC")
ax.errorbar(lb_plot["test AUC"], pos, xerr=[lb_plot["test AUC"] - lb_plot.ci_low, lb_plot.ci_high - lb_plot["test AUC"]], fmt="D", color="black", capsize=2, label="test AUC (95% CI)")
ax.set_yticks(pos)
ax.set_yticklabels(lb_plot.model + " | " + lb_plot["feature group"].str.split(": ").str[0], fontsize=8)
ax.axvline(0.5, color="gray", ls="--", lw=1)
ax.set(xlim=(0.45, 0.88), xlabel="AUC", title="All models: validation AUC (bars) and test AUC (diamonds)\nblue = main classic, green = extra classic, red = neural network")
ax.legend(loc="lower right")
plt.savefig(OUT / "08c_all_models_leaderboard.png", bbox_inches="tight")
plt.show()
n_tried = len(set(ex_val.model)) + len(MODELS) + 3
n_ex_win = int(ex_val.better_convincingly.sum())
print(f"{n_tried} model families were compared. Extra models that beat the best of LR/RF/HGB convincingly (same features): {n_ex_win} of {len(ex_val)}. "
      f"Test AUCs of the best configuration of every family range from {leaderboard['test AUC'].min():.2f} to {leaderboard['test AUC'].max():.2f}.")

## 17d. Do cast and crew track records help? (a separate experiment)
Section 5b built leak-free track-record features for the director, writer, top cast, technical team, art director, production company and producer. Here we test whether they improve the predictions, **without touching the main comparison above**:
* **Group E** = base + director/writer/cast records (compare with **A**, base only). **Group F** = base + Wikipedia + YouTube + director/writer/cast records (compare with **D**, the same without them).
* **Group G** = base + technical team and art director records (compare with **A**). **Group H** = base + Wikipedia + YouTube + those records (compare with **D**).
* Same chronological outer folds, same inner tuning, same models (the three main ones plus XGBoost, the strongest extra model).
* **Group I** = base + production company and producer records (compare with **A**). **Group J** = base + Wikipedia + YouTube + those records (compare with **D**). The company and producer names come from each movie's Wikipedia infobox (section 5b).
* The gain is the **paired AUC difference** on the same validation movies, with a bootstrap interval and a Benjamini-Hochberg-adjusted p-value over all comparisons.

**What to expect.** Only 854 movies are labelled, so most directors and actors have no earlier labelled film. If track records help, the gain will be small; a null result is a valid finding and is reported as such.

In [ ]:
ANY_MODELS = {**MODELS, **EXTRA}
CREW_MODELS = list(MODELS) + (["XGBoost"] if "XGBoost" in EXTRA else [])
gA, gD = list(GROUPS)[0], list(GROUPS)[-1]
gE, gF, gG, gH, gI, gJ = list(CREW_GROUPS)
COMPARISONS = [(gE, gA), (gF, gD), (gG, gA), (gH, gD), (gI, gA), (gJ, gD)]


def tune_any(X, y, model_key, group, n_inner=3):
    est, grid, scale = ANY_MODELS[model_key]
    pipe = Pipeline([("prep", make_preprocessor(group, scale)), ("model", est())])
    return GridSearchCV(pipe, grid, scoring="roc_auc", cv=TimeSeriesSplit(n_splits=n_inner), n_jobs=1, refit=True).fit(X, y)


def crew_fold(model_key, group, k, tr, va):
    return model_key, group, k, va, tune_any(dev.iloc[tr], y_dev[tr], model_key, group).predict_proba(dev.iloc[va])[:, 1]


t_crew = time.time()
out = Parallel(n_jobs=-1)(delayed(crew_fold)(m, g, k, tr, va) for m in CREW_MODELS for g in CREW_GROUPS for k, (tr, va) in enumerate(outer_folds))
crew_oof = {}
for m, g, k, va, p in out:
    crew_oof.setdefault((m, g), np.full(len(dev), np.nan))[va] = p
print(f"Cast, crew and technical-team groups validated in {time.time() - t_crew:.0f} s ({len(out)} outer fits).")


def oof_of(m, g):
    return crew_oof[(m, g)] if (m, g) in crew_oof else (oof[(m, g)] if m in MODELS else ex_oof[(m, g)])


rows = []
for m in CREW_MODELS:
    for new, old in COMPARISONS:
        d, lo, hi, p = paired_boot(y_dev[val_idx], oof_of(m, new)[val_idx], oof_of(m, old)[val_idx])
        rows.append({"model": m, "with cast/crew": new.split(":")[0], "compared with": old.split(":")[0], "AUC without": auc_fast(y_dev[val_idx], oof_of(m, old)[val_idx]),
                     "AUC with": auc_fast(y_dev[val_idx], oof_of(m, new)[val_idx]), "gain": d, "ci_low": lo, "ci_high": hi, "p_bootstrap": p})
crew_val = pd.DataFrame(rows)
crew_val["p_adjusted_BH"] = bh_adjust(crew_val.p_bootstrap.values)
crew_val["convincing_gain"] = (crew_val.ci_low > 0) & (crew_val.p_adjusted_BH < 0.05)
print("Validation (out-of-fold) effect of adding cast/crew track records:")
display(crew_val.round(3))
n_conv, n_cmp = int(crew_val.convincing_gain.sum()), len(crew_val)
n_pos = int((crew_val.gain > 0).sum())
print(f"Convincing gains: {n_conv} of {n_cmp}. Positive point estimates: {n_pos} of {n_cmp}. Mean gain {crew_val.gain.mean():+.3f}.")

# One test scoring per model and comparison (supplementary; nothing is selected with it).
crew_test_cache = {}


def test_pred(m, g):
    if (m, g) not in crew_test_cache:
        crew_test_cache[(m, g)] = (final_fit(m, g) if (m in MODELS and g in GROUPS) else tune_any(dev, y_dev, m, g, n_inner=N_OUTER)).predict_proba(test)[:, 1]
    return crew_test_cache[(m, g)]


rows = []
for m in CREW_MODELS:
    for new, old in COMPARISONS:
        d, lo, hi, p = paired_boot(y_test, test_pred(m, new), test_pred(m, old))
        rows.append({"model": m, "with cast/crew": new.split(":")[0], "compared with": old.split(":")[0], "test AUC without": auc_fast(y_test, test_pred(m, old)),
                     "test AUC with": auc_fast(y_test, test_pred(m, new)), "gain": d, "ci_low": lo, "ci_high": hi, "p_bootstrap": p})
crew_test = pd.DataFrame(rows)
print("Test period (one scoring, supplementary):")
display(crew_test.round(3))

fig, ax = plt.subplots(1, 2, figsize=(13, 10), sharey=True)
for a, (tab, ttl) in zip(ax, [(crew_val, "Validation (out-of-fold)"), (crew_test, "Test (one scoring)")]):
    lbl = [f"{r.model}\n{r['with cast/crew']} vs {r['compared with']}" for _, r in tab.iterrows()]
    y_pos = np.arange(len(tab))
    a.errorbar(tab.gain, y_pos, xerr=[tab.gain - tab.ci_low, tab.ci_high - tab.gain], fmt="o", color="#4C72B0", capsize=3)
    a.axvline(0, color="gray", ls="--")
    a.set_yticks(y_pos)
    a.set_yticklabels(lbl, fontsize=8)
    a.set(title=f"AUC gain from cast/crew track records\n{ttl}", xlabel="AUC with minus AUC without (95% CI)")
    a.invert_yaxis()
plt.tight_layout()
plt.savefig(OUT / "11_crew_track_records.png", bbox_inches="tight")
plt.show()

verdict = ("Cast, crew, studio and producer track records give a **convincing** gain on validation" if n_conv >= 1 else "Cast, crew, studio and producer track records do **not** give a convincing gain on validation")
print(f"\n{verdict} ({n_conv} of {n_cmp} comparisons; mean gain {crew_val.gain.mean():+.3f} AUC). On the test period the mean gain is {crew_test.gain.mean():+.3f} AUC "
      f"({int((crew_test.ci_low > 0).sum())} of {len(crew_test)} intervals above zero). They are reported, not adopted: the main results above do not use them.")
results["crew_experiment"] = {"validation": crew_val.round(4).to_dict("records"), "test": crew_test.round(4).to_dict("records"), "convincing_gains": n_conv, "comparisons": n_cmp,
                              "mean_val_gain": float(crew_val.gain.mean()), "mean_test_gain": float(crew_test.gain.mean())}

## 17e. Are the predicted probabilities trustworthy? Calibration and SHAP
**Calibration.** A model is *calibrated* if, among movies it gives a 40% chance of being a Hit, about 40% really are Hits. A **reliability curve** splits movies into five equal-sized groups by predicted probability and compares the average prediction with the real Hit rate. The **Brier score** (lower is better) is the average squared error of the probabilities; the reference is always predicting the development Hit rate. Calibration matters because the Hit rate fell from the development period to the test period, so models trained on the past are expected to over-predict Hits.

**SHAP.** SHAP values split each prediction into the contribution of every feature (positive pushes towards Hit). They are computed for the tuned XGBoost model on the **development data only**, so the test period plays no role. SHAP, like permutation importance, describes what the *model* uses, not what causes success.

In [ ]:
from sklearn.calibration import calibration_curve

cal_models = [("Chosen model: " + CH_MODEL, CH_MODEL, CH_GROUP)]
if "XGBoost" in EXTRA:
    xb = ex_best.loc["XGBoost", "feature group"]
    cal_models.append(("XGBoost", "XGBoost", xb))
base_rate_dev = y_dev.mean()
cal_rows, curves = [], {}
for label, m, g in cal_models:
    pv = (oof[(m, g)] if m in MODELS else ex_oof[(m, g)])[val_idx]
    pt = test_pred(m, g)
    for split, y_, p_ in [("validation (out-of-fold)", y_dev[val_idx], pv), ("test", y_test, pt)]:
        frac, mean_p = calibration_curve(y_, p_, n_bins=5, strategy="quantile")
        curves[(label, split)] = (mean_p, frac)
        brier = brier_score_loss(y_, np.clip(p_, 0, 1))
        ref = brier_score_loss(y_, np.full(len(y_), base_rate_dev))
        cal_rows.append({"model": label, "feature group": g.split(":")[0], "period": split, "mean predicted P(Hit)": p_.mean(), "actual Hit rate": y_.mean(),
                         "Brier": brier, "Brier of always predicting the development rate": ref, "Brier skill": 1 - brier / ref,
                         "expected calibration error": float(np.mean(np.abs(mean_p - frac)))})
cal_table = pd.DataFrame(cal_rows)
display(cal_table.round(3))

fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
for a, split in zip(ax, ["validation (out-of-fold)", "test"]):
    a.plot([0, 1], [0, 1], "k--", lw=1, label="perfectly calibrated")
    for (label, s), (mp, fr) in curves.items():
        if s == split:
            a.plot(mp, fr, "o-", label=label)
    a.set(title=f"Reliability curve: {split}", xlabel="Mean predicted probability of Hit", ylabel="Actual share of Hits", xlim=(0, 1), ylim=(0, 1))
    a.legend(fontsize=8)
plt.tight_layout()
plt.savefig(OUT / "12_calibration.png", bbox_inches="tight")
plt.show()

shift = cal_table[cal_table.period == "test"].iloc[0]
print(f"On the test period the chosen model predicts a mean Hit probability of {shift['mean predicted P(Hit)']:.2f} while the actual Hit rate is {shift['actual Hit rate']:.2f}: "
      f"{'it over-predicts Hits, as expected after the fall in the Hit rate' if shift['mean predicted P(Hit)'] > shift['actual Hit rate'] + 0.03 else 'the average level is close'}.")
print("Use the predicted probabilities for ranking movies; for decisions that need absolute probabilities they should be recalibrated on recent data.")
results["calibration"] = cal_table.round(4).to_dict("records")

In [ ]:
# SHAP for the tuned XGBoost (or the Random forest if XGBoost is not available), explained on the development data only.
try:
    try:
        import shap
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "shap"])
        import shap
    shap_model = "XGBoost" if "XGBoost" in EXTRA else "Random forest"
    shap_group = xb if shap_model == "XGBoost" else CH_GROUP
    gs_shap = tune_any(dev, y_dev, shap_model, shap_group, n_inner=N_OUTER)
    prep, mdl = gs_shap.best_estimator_.named_steps["prep"], gs_shap.best_estimator_.named_steps["model"]
    X_dev_t = prep.transform(dev)
    names = [n.split("__", 1)[-1] for n in prep.get_feature_names_out()]
    sv = shap.TreeExplainer(mdl).shap_values(X_dev_t)
    sv = sv[1] if isinstance(sv, list) else (sv[..., 1] if getattr(sv, "ndim", 2) == 3 else sv)
    shap_table = pd.DataFrame({"feature": names, "mean |SHAP|": np.abs(sv).mean(axis=0)}).sort_values("mean |SHAP|", ascending=False).reset_index(drop=True)
    print(f"SHAP for {shap_model} on '{shap_group.split(': ')[1]}', development data only. Ten most influential features:")
    display(shap_table.head(10).round(4))
    plt.figure()
    shap.summary_plot(sv, X_dev_t, feature_names=names, max_display=12, show=False)
    plt.title(f"SHAP summary: {shap_model} ({shap_group.split(':')[0]}), development data")
    plt.savefig(OUT / "13_shap_summary.png", bbox_inches="tight")
    plt.show()
    results["shap"] = {"model": shap_model, "group": shap_group, "top10": shap_table.head(10).round(4).to_dict("records")}
except Exception as err:                                               # never let an optional explanation stop the pipeline
    shap_table = pd.DataFrame({"note": [f"SHAP skipped: {type(err).__name__}: {err}"]})
    print(f"SHAP could not be computed here and is skipped: {type(err).__name__}: {err}")

## 17f. Follow-up checks: recalibration, "would more data help?", and YouTube timing
Three questions raised by the earlier results. All are **supplementary**: nothing here changes the models chosen above.

**1. Recalibration on recent movies.** The Hit rate fell after 2022, so the models over-predict Hits on the test period. We try three fixes and judge them on movies they never saw: the test period is split by date into an **earlier half** (used only to fit the fix, as if those movies had just become labelled) and a **later half** (used only to judge). The fixes are (a) a logistic "Platt" curve fitted on **development** predictions only, (b) a single **level shift** fitted on the earlier test half, and (c) a Platt curve fitted on the earlier test half. Recalibration keeps the ranking of movies, so the AUC is unchanged; only the probabilities move. This is a post-hoc analysis of the test period with about 138 judging movies, so intervals are wide.

**2. Would more data help?** We train the chosen model and logistic regression on the most recent 100, 200, 300, 400 and all 482 movies before the last validation window, and score the same 96 validation movies each time (hyperparameters tuned inside each training set). If the curve keeps rising, more labelled movies are worth collecting.

**3. YouTube timing.** The file has no collection date. If comment likes were read after release, older movies would have had longer to collect likes, so likes per comment would **rise with the movie's age**. We test that, and also check whether the models get worse when comment likes are removed.

In [ ]:
from scipy.optimize import brentq
from scipy.special import expit


def logit(p):
    p = np.clip(p, 1e-4, 1 - 1e-4)
    return np.log(p / (1 - p))


def brier(y, p):
    return float(np.mean((np.asarray(p) - np.asarray(y)) ** 2))


def ece_q(y, p, bins=4):
    frac, mp = calibration_curve(y, p, n_bins=bins, strategy="quantile")
    return float(np.mean(np.abs(frac - mp)))


def brier_gain_ci(y, p_new, p_old, n=2000, seed=SEED):
    """Paired bootstrap of the Brier improvement (old minus new; positive = better)."""
    y, p_new, p_old = np.asarray(y), np.asarray(p_new), np.asarray(p_old)
    rg = np.random.default_rng(seed)
    d = []
    for _ in range(n):
        i = rg.integers(0, len(y), len(y))
        d.append(brier(y[i], p_old[i]) - brier(y[i], p_new[i]))
    return brier(y, p_old) - brier(y, p_new), *np.percentile(d, [2.5, 97.5])


half = len(test) // 2
cal_i, ev_i = np.arange(half), np.arange(half, len(test))
y_ev = y_test[ev_i]
print(f"Fit window: {half} test movies ({test.release_date.iloc[cal_i].min().date()} to {test.release_date.iloc[cal_i].max().date()}, Hit rate {y_test[cal_i].mean():.2f}). "
      f"Judging window: {len(ev_i)} test movies ({test.release_date.iloc[ev_i].min().date()} to {test.release_date.iloc[ev_i].max().date()}, Hit rate {y_ev.mean():.2f}).")
recal_rows, recal_curves = [], {}
for label, m, g in cal_models:
    pt = test_pred(m, g)
    pv = oof_of(m, g)[val_idx]
    platt_dev = LogisticRegression(C=1e6, max_iter=1000).fit(logit(pv)[:, None], y_dev[val_idx])
    shift = brentq(lambda b: expit(logit(pt[cal_i]) + b).mean() - y_test[cal_i].mean(), -6, 6)
    platt_rec = LogisticRegression(C=1e6, max_iter=1000).fit(logit(pt[cal_i])[:, None], y_test[cal_i])
    variants = {"none (as trained)": pt[ev_i],
                "(a) Platt, fitted on development predictions": platt_dev.predict_proba(logit(pt[ev_i])[:, None])[:, 1],
                "(b) level shift, fitted on earlier test half": expit(logit(pt[ev_i]) + shift),
                "(c) Platt, fitted on earlier test half": platt_rec.predict_proba(logit(pt[ev_i])[:, None])[:, 1]}
    for name, p_ in variants.items():
        gain, lo, hi = brier_gain_ci(y_ev, p_, variants["none (as trained)"]) if name != "none (as trained)" else (0.0, np.nan, np.nan)
        recal_rows.append({"model": label, "recalibration": name, "mean predicted P(Hit)": p_.mean(), "actual Hit rate": y_ev.mean(), "Brier": brier(y_ev, p_), "Brier gain vs none": gain,
                           "gain ci_low": lo, "gain ci_high": hi, "calibration error (4 bins)": ece_q(y_ev, p_), "AUC": auc_fast(y_ev, p_)})
        if label == cal_models[0][0]:
            recal_curves[name] = calibration_curve(y_ev, p_, n_bins=4, strategy="quantile")
recal_table = pd.DataFrame(recal_rows)
display(recal_table.round(3))
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].plot([0, 1], [0, 1], "k--", lw=1)
for name, (fr, mp) in recal_curves.items():
    ax[0].plot(mp, fr, "o-", label=name)
ax[0].set(title=f"Reliability on the later test half\n({cal_models[0][0]})", xlabel="Mean predicted probability of Hit", ylabel="Actual share of Hits", xlim=(0, 1), ylim=(0, 1))
ax[0].legend(fontsize=7)
piv = recal_table.pivot(index="recalibration", columns="model", values="Brier").loc[list(variants)]
for j, col in enumerate(piv.columns):
    ax[1].plot(piv[col].values, np.arange(len(piv)) + (j - 0.5) * 0.25, "o", label=col)
ax[1].set_yticks(np.arange(len(piv)))
ax[1].set_yticklabels(piv.index, fontsize=8)
ax[1].set(title="Brier score on the later test half (lower is better)", xlabel="Brier (axis does not start at 0)")
ax[1].invert_yaxis()
ax[1].legend(fontsize=8, loc="lower right")
plt.tight_layout()
plt.savefig(OUT / "14_recalibration.png", bbox_inches="tight")
plt.show()
best_fix = recal_table[(recal_table.model == cal_models[0][0]) & (recal_table.recalibration != "none (as trained)")].sort_values("Brier").iloc[0]
print(f"For {cal_models[0][0]}, the best fix on the later test half is '{best_fix.recalibration}': Brier {best_fix.Brier:.3f} against {recal_table[(recal_table.model == cal_models[0][0]) & (recal_table.recalibration == 'none (as trained)')].Brier.iloc[0]:.3f} "
      f"without it (gain {best_fix['Brier gain vs none']:+.3f}, 95% CI {best_fix['gain ci_low']:+.3f} to {best_fix['gain ci_high']:+.3f}); mean prediction {best_fix['mean predicted P(Hit)']:.2f} against an actual rate of {best_fix['actual Hit rate']:.2f}.")
results["recalibration"] = recal_table.round(4).to_dict("records")

In [ ]:
# 2. Would more data help? Learning curve on a fixed, later validation window.
tr_all, va_last = outer_folds[-1]
sizes = [100, 200, 300, 400, len(tr_all)]
lc_rows = []
for m in dict.fromkeys([CH_MODEL, "Logistic regression"]):
    for n_tr in sizes:
        idx = tr_all[-n_tr:]                                   # the most recent n_tr movies before the validation window
        gs = tune(dev.iloc[idx], y_dev[idx], m, CH_GROUP, n_inner=3)
        p_ = gs.predict_proba(dev.iloc[va_last])[:, 1]
        lo, hi = boot_auc_ci(y_dev[va_last], p_, n=1000)
        lc_rows.append({"model": m, "training movies": n_tr, "validation AUC": auc_fast(y_dev[va_last], p_), "ci_low": lo, "ci_high": hi})
lc_table = pd.DataFrame(lc_rows)
display(lc_table.round(3))
lc_main = lc_table[lc_table.model == CH_MODEL].set_index("training movies")["validation AUC"]
print(f"{CH_MODEL}: validation AUC {lc_main.iloc[0]:.3f} with {lc_main.index[0]} training movies, {lc_main.loc[200]:.3f} with 200, {lc_main.iloc[-1]:.3f} with {lc_main.index[-1]}. "
      f"Gain from 200 to {lc_main.index[-1]} movies: {lc_main.iloc[-1] - lc_main.loc[200]:+.3f} AUC. The {len(va_last)}-movie validation window is small, so differences of about 0.05 are within noise.")
results["learning_curve"] = lc_table.round(4).to_dict("records")

In [ ]:
# 3. YouTube timing: do comment likes grow with the movie's age?
yt = merged[(merged.in_yt_file == 1) & (merged.cm_pre_n > 0) & merged.release_date.notna()].copy()
yt["age_years"] = (LAST_RELEASE - yt.release_date).dt.days / 365.25
yt["likes_per_comment"] = yt.cm_pre_likes / yt.cm_pre_n
rho_l, p_l = stats.spearmanr(yt.age_years, yt.likes_per_comment)
rho_n, p_n = stats.spearmanr(yt.age_years, yt.cm_pre_n)
yt_year_table = yt.groupby(yt.release_date.dt.year).agg(movies=("tmdb_id", "size"), median_comments_before_release=("cm_pre_n", "median"), median_likes_per_comment=("likes_per_comment", "median")).reset_index().rename(columns={"release_date": "release year"})
display(yt_year_table.round(2))
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
for m, d_ in lc_table.groupby("model"):
    ax[0].errorbar(d_["training movies"], d_["validation AUC"], yerr=[d_["validation AUC"] - d_.ci_low, d_.ci_high - d_["validation AUC"]], marker="o", capsize=3, label=m)
ax[0].axhline(0.5, color="gray", ls="--")
ax[0].set(title=f"Learning curve (same {len(va_last)} validation movies)", xlabel="Number of training movies", ylabel="Validation AUC (95% CI)")
ax[0].legend(fontsize=8)
ax[1].bar(yt_year_table["release year"].astype(str), yt_year_table.median_likes_per_comment, color="#4C72B0")
ax[1].set(title="YouTube: median likes per pre-release comment, by release year", xlabel="Release year", ylabel="Likes per comment")
plt.tight_layout()
plt.savefig(OUT / "15_learning_curve_and_youtube_timing.png", bbox_inches="tight")
plt.show()
rising = rho_l > 0.2 and p_l < 0.05
print(f"Spearman correlation between a movie's age and its likes per pre-release comment: {rho_l:+.2f} (p = {p_l:.3g}, {len(yt)} movies); with the number of pre-release comments: {rho_n:+.2f} (p = {p_n:.3g}).")
print("Reading: " + ("comment likes grow clearly with age, which suggests they were read AFTER release, so they may carry post-release information." if rising else
                     "there is no strong rise of likes per comment with age, so this check does not show that comment likes were collected long after release (it cannot prove they were collected before it)."))

yt_models = list(dict.fromkeys([CH_MODEL, "XGBoost"] if "XGBoost" in EXTRA else [CH_MODEL]))
gDm = "D-: base + Wikipedia + YouTube without comment likes"
out = Parallel(n_jobs=-1)(delayed(crew_fold)(m, gDm, k, tr, va) for m in yt_models for k, (tr, va) in enumerate(outer_folds))
nl_oof = {}
for m, g, k, va, p in out:
    nl_oof.setdefault((m, g), np.full(len(dev), np.nan))[va] = p
rows = []
for m in yt_models:
    d, lo, hi, p = paired_boot(y_dev[val_idx], nl_oof[(m, gDm)][val_idx], oof_of(m, gD)[val_idx])
    dt, lot, hit, _ = paired_boot(y_test, test_pred(m, gDm), test_pred(m, gD))
    rows.append({"model": m, "validation AUC with comment likes (D)": auc_fast(y_dev[val_idx], oof_of(m, gD)[val_idx]), "validation AUC without comment likes": auc_fast(y_dev[val_idx], nl_oof[(m, gDm)][val_idx]),
                 "change without likes (validation)": d, "ci_low": lo, "ci_high": hi, "change without likes (test)": dt, "test ci_low": lot, "test ci_high": hit})
yt_nolikes = pd.DataFrame(rows)
display(yt_nolikes.round(3))
print("Removing comment likes " + ("does not hurt the models: the likes carry no detectable extra signal, so any post-release leakage through them would not be driving the results." if (yt_nolikes["change without likes (validation)"] > -0.01).all() else
      "lowers validation AUC for at least one model, so comment likes carry signal (which may be genuine or may be post-release information; the data cannot tell)."))
results["youtube_timing"] = {"spearman_age_vs_likes_per_comment": float(rho_l), "p_value": float(p_l), "n_movies": int(len(yt)), "spearman_age_vs_comment_count": float(rho_n),
                             "without_comment_likes": yt_nolikes.round(4).to_dict("records")}

## 17g. Currency and basis check (using the Wikipedia infobox text)
The two data files have no currency column, so we cannot say directly which currency the budget and revenue are in. The infobox text read in section 5b gives a way to check: it states budget and box office in the original currency (mostly Indian rupees), so for movies where **both** figures are given in rupees we can
* compute the **implied exchange rate** (dollars in the file per rupee in the text) for each data source, to see whether the file really is in one currency, and
* compute a **currency-free ROI** (box office divided by budget, both in rupees) and see how often it gives the same Hit/Flop label as the file's ROI.

This is a diagnostic. It does not change any model, and the infobox figures are themselves reported by Wikipedia editors (gross versus net is often unclear).

In [ ]:
_UNIT = {"crore": 1e7, "cr": 1e7, "lakh": 1e5, "million": 1e6, "billion": 1e9, "m": 1e6}


def _money(s):
    """'₹239–240 crore' -> ('INR', 2.395e9); returns (None, nan) when there is no currency symbol or no unit."""
    if not isinstance(s, str) or not s:
        return None, np.nan
    t = s.replace(",", "")
    cur = "INR" if "₹" in t else ("USD" if "$" in t else None)
    if cur is None:
        return None, np.nan
    m = re.search(r"([\d.]+)(?:\s*[–-]\s*([\d.]+))?\s*(crore|cr|lakh|million|billion)\b", t[t.index("₹" if cur == "INR" else "$") + 1:], flags=re.I)
    if not m:
        return None, np.nan
    try:
        a = float(m.group(1))
        b = float(m.group(2)) if m.group(2) else a
    except ValueError:
        return None, np.nan
    return cur, (a + b) / 2 * _UNIT[m.group(3).lower()]


ib = infobox.drop_duplicates("wiki_title").set_index("wiki_title")
cc = lab[["wiki_title", "year", "budget", "revenue", "roi", "hit", "budget_source", "revenue_source"]].copy()
cc = cc[cc.wiki_title.isin(ib.index)]
cc[["b_cur", "b_txt"]] = [_money(ib.budget_text.get(t, "")) for t in cc.wiki_title]
cc[["g_cur", "g_txt"]] = [_money(ib.gross_text.get(t, "")) for t in cc.wiki_title]
cc["source"] = np.where((cc.budget_source == "tmdb") & (cc.revenue_source == "tmdb"), "tmdb/tmdb", "has a Wikipedia value")
inr = cc[(cc.b_cur == "INR") & (cc.g_cur == "INR") & (cc.b_txt > 0) & (cc.g_txt > 0)].copy()
inr["usd_per_inr_budget"] = inr.budget / inr.b_txt
inr["usd_per_inr_revenue"] = inr.revenue / inr.g_txt
inr["roi_text"] = inr.g_txt / inr.b_txt
inr["hit_text"] = (inr.roi_text >= 2).astype(int)
inr["same_label"] = (inr.hit_text == inr.hit).astype(int)
print(f"Movies with both budget and box office stated in rupees on Wikipedia: {len(inr)} of {len(cc)} labelled movies with an infobox.")

rate_tab = inr.groupby("source").agg(movies=("hit", "size"), median_usd_per_inr_budget=("usd_per_inr_budget", "median"), q25=("usd_per_inr_budget", lambda s: s.quantile(.25)),
                                     q75=("usd_per_inr_budget", lambda s: s.quantile(.75)), median_usd_per_inr_revenue=("usd_per_inr_revenue", "median"),
                                     hit_rate_file=("hit", "mean"), hit_rate_rupee_roi=("hit_text", "mean"), same_label=("same_label", "mean")).round(4).reset_index()
print("Implied exchange rate and agreement of the Hit label with a currency-free ROI, by data source:")
display(rate_tab)
yr_tab = inr.groupby("year").agg(movies=("hit", "size"), median_usd_per_inr=("usd_per_inr_budget", "median")).round(4).reset_index()
display(yr_tab.T)
mixed = cc[(cc.budget_source != cc.revenue_source)]
band = (mixed.roi.between(2 / 1.15, 2 * 1.15)).sum()
q = inr.usd_per_inr_budget
print(f"Median implied rate {q.median():.4f} dollars per rupee (middle half {q.quantile(.25):.4f} to {q.quantile(.75):.4f}); real rates fell from about 0.0155 in 2015 to about 0.0115 in 2025, so the file looks like US dollars converted from rupees.")
print(f"Budget and revenue from different sources: {len(mixed)} labelled movies; {int(band)} of them have an ROI within 15% of the Hit threshold, so a rate difference between sources could flip only about that many labels.")
agree = inr.same_label.mean()
print(f"The Hit label from the rupee figures alone agrees with the file's label for {agree:.1%} of these {len(inr)} movies.")
currency_check = rate_tab
results["currency_check"] = {"n_rupee_pairs": int(len(inr)), "median_usd_per_inr": float(q.median()), "q25": float(q.quantile(.25)), "q75": float(q.quantile(.75)),
                             "label_agreement": float(agree), "mixed_source_movies": int(len(mixed)), "mixed_near_threshold": int(band),
                             "by_source": rate_tab.to_dict("records")}

## 18. Bonus: predicting how much a movie earns (revenue regression)
The Hit label divides revenue by budget and hides size. Here we predict **log10 of revenue** itself with **Ridge regression** and a **Random Forest regressor**, using the same four feature groups,
the same pipelines (genres chosen on training rows only) and the same **nested chronological validation**. Models are selected by the lowest validation **RMSE**; the test period is used once at the end.
Metrics: **RMSE** and **MAE** (in log10 units; an error of 0.30 means a typical miss by a factor of 2) and **R-squared** (share of variance explained; 0 = no better than predicting the average).

In [ ]:
REG = {
    "Ridge": (lambda: Ridge(), {"model__alpha": [0.3, 1, 3, 10, 30, 100]}, True),
    "Random forest regressor": (lambda: RandomForestRegressor(n_estimators=200, random_state=SEED, n_jobs=1), {"model__min_samples_leaf": [3, 8], "model__max_features": [0.5, 1.0]}, False),
}
y_rev, y_rev_test = np.log10(dev.revenue.values), np.log10(test.revenue.values)


def make_reg_pipe(mk, group):
    est, _, scale = REG[mk]
    return Pipeline([("prep", make_preprocessor(group, scale)), ("model", est())])


def reg_tune(X, y, mk, group, n_inner=3):
    return GridSearchCV(make_reg_pipe(mk, group), REG[mk][1], scoring="neg_root_mean_squared_error", cv=TimeSeriesSplit(n_splits=n_inner), n_jobs=1, refit=True).fit(X, y)


def reg_fold(mk, group, tr, va):
    gs = reg_tune(dev.iloc[tr], y_rev[tr], mk, group)
    return mk, group, va, gs.predict(dev.iloc[va])


reg_out = Parallel(n_jobs=-1)(delayed(reg_fold)(mk, g, tr, va) for mk in REG for g in GROUPS for tr, va in outer_folds)
reg_oof = {}
for mk, g, va, p in reg_out:
    reg_oof.setdefault((mk, g), np.full(len(dev), np.nan))[va] = p
rmse = lambda a, b: float(np.sqrt(mean_squared_error(a, b)))
rrows = []
for (mk, g), p in reg_oof.items():
    fr = [rmse(y_rev[va], p[va]) for _, va in outer_folds]
    rrows.append({"feature group": g, "model": mk, "cv_RMSE_mean": np.mean(fr), "cv_RMSE_sd": np.std(fr, ddof=1), "pooled R2": r2_score(y_rev[val_idx], p[val_idx]),
                  "pooled MAE": mean_absolute_error(y_rev[val_idx], p[val_idx]), "pooled RMSE": rmse(y_rev[val_idx], p[val_idx])})
dfr = [rmse(y_rev[va], np.full(len(va), y_rev[tr].mean())) for tr, va in outer_folds]
dummy_pred = np.concatenate([np.full(len(va), y_rev[tr].mean()) for tr, va in outer_folds])
rrows.insert(0, {"feature group": "(none)", "model": "DummyRegressor (training mean)", "cv_RMSE_mean": np.mean(dfr), "cv_RMSE_sd": np.std(dfr, ddof=1), "pooled R2": r2_score(y_rev[val_idx], dummy_pred),
                 "pooled MAE": mean_absolute_error(y_rev[val_idx], dummy_pred), "pooled RMSE": rmse(y_rev[val_idx], dummy_pred)})
reg_cv = pd.DataFrame(rrows)
print("Validation results (out-of-fold, development period). Lower RMSE and MAE are better; selection uses cv_RMSE_mean.")
display(reg_cv.round(3))
results["regression_validation"] = reg_cv.round(4).to_dict("records")

real_r = reg_cv[reg_cv.model != "DummyRegressor (training mean)"]
best_reg = real_r.loc[real_r.groupby("model").cv_RMSE_mean.idxmin()].set_index("model")
chosen_reg = real_r.loc[real_r.cv_RMSE_mean.idxmin()]
tr_rows = []
reg_test_preds = {}
for mk in REG:
    g = best_reg.loc[mk, "feature group"]
    gs = reg_tune(dev, y_rev, mk, g, n_inner=N_OUTER)
    p = gs.predict(test)
    reg_test_preds[mk] = p
    tr_rows.append({"model": mk, "feature group": g, "R2": r2_score(y_rev_test, p), "MAE (log10)": mean_absolute_error(y_rev_test, p), "RMSE (log10)": rmse(y_rev_test, p), "typical error factor": 10 ** mean_absolute_error(y_rev_test, p),
                    "chosen by validation": "yes" if mk == chosen_reg.model else ""})
pd_ = np.full(len(test), y_rev.mean())
tr_rows.append({"model": "DummyRegressor (training mean)", "feature group": "(none)", "R2": r2_score(y_rev_test, pd_), "MAE (log10)": mean_absolute_error(y_rev_test, pd_), "RMSE (log10)": rmse(y_rev_test, pd_),
                "typical error factor": 10 ** mean_absolute_error(y_rev_test, pd_), "chosen by validation": ""})
reg_test = pd.DataFrame(tr_rows)
print("FINAL TEST RESULTS for revenue regression (each model with its own validation-chosen feature group):")
display(reg_test.round(3))
results["regression_test"] = reg_test.round(4).to_dict("records")
best_r2 = reg_test[reg_test.model != "DummyRegressor (training mean)"].R2.max()
print(f"Why this is harder than Hit or Flop: revenue spans several orders of magnitude and depends on marketing, competition, reviews and word of mouth, none of which are known before release; "
      f"Hit or Flop only needs the sign of the result relative to the budget. The best test R-squared is {best_r2:.2f}, and the typical error is a factor of about {reg_test['typical error factor'].min():.1f}.")
fig, ax = plt.subplots(figsize=(5.2, 5))
ax.scatter(y_rev_test, reg_test_preds[chosen_reg.model], s=14, alpha=0.6)
lim = [min(y_rev_test.min(), reg_test_preds[chosen_reg.model].min()), max(y_rev_test.max(), reg_test_preds[chosen_reg.model].max())]
ax.plot(lim, lim, "k--")
ax.set(title=f"Test revenue: {chosen_reg.model}", xlabel="actual log10(revenue)", ylabel="predicted log10(revenue)")
plt.savefig(OUT / "10_revenue_regression.png", bbox_inches="tight")
plt.show()

## 19. Conclusions (generated from the computed results)
Every sentence below is built from the numbers above when the notebook runs, so the text cannot drift from the results.
Findings are labelled **supported** (the evidence is statistically convincing), **suggestive** (a direction is visible but the interval includes no effect) or **not supported**.

In [ ]:
ch = results["final_test"][CH_MODEL]
ci = ch["ci"]["auc"]
bal_acc = (ch["hit_recall"] + ch["flop_recall"]) / 2
maj_acc = 1 - y_test.mean()
lines = []

# Q1
verdict = "Yes, but only moderately" if ci[0] > 0.5 else "Not reliably"
higher = ch["auc"] > chosen_row.pooled_oof_auc
acc_gap = ch["accuracy"] - maj_acc
lines.append(("1. Can we predict Hit or Flop before release?",
              f"{verdict}. The frozen {CH_MODEL} (features: {CH_GROUP.split(': ')[1]}) reaches test AUC {ch['auc']:.2f} (95% CI {ci[0]:.2f} to {ci[1]:.2f}) against 0.50 for the dummy baseline, so it ranks Hits above Flops clearly better than chance. "
              f"At its validation-chosen threshold it finds {ch['hit_recall']:.0%} of Hits and {ch['flop_recall']:.0%} of Flops (balanced accuracy {bal_acc:.0%}); overall accuracy is {ch['accuracy']:.0%} versus {maj_acc:.0%} for always predicting Flop, "
              f"which is only {'slightly' if acc_gap < 0.05 else 'clearly'} better. Test AUC is {'higher' if higher else 'lower'} than validation AUC ({ch['auc']:.2f} versus {chosen_row.pooled_oof_auc:.2f}). "
              + ("A likely reason is that the final model learns from all development movies, while the early validation folds trained on as few as 98; the intervals overlap, so the difference should not be over-read."
                 if higher else f"The market shift (Hit rate {dev.hit.mean():.0%} to {test.hit.mean():.0%}) is a likely contributor.")))
# Q2
tm_txt = "; ".join(f"{mk}: test AUC {v['auc']:.2f} ({v['ci']['auc'][0]:.2f} to {v['ci']['auc'][1]:.2f})" for mk, v in results["final_test"].items())
tied = not model_pairs.distinguishable.any()
lines.append(("2. Which model performs best?",
              f"By cross-validation the best combination is {CH_MODEL} on '{CH_GROUP.split(': ')[1]}'. Test results: {tm_txt}. "
              + ("No pair of model types is statistically distinguishable on the validation predictions (all paired intervals include 0), so the ranking should be read as a tie and the simplest model is a reasonable practical choice."
                 if tied else "At least one pair of model types differs clearly on validation; see the paired model table.")))
# Q2b
dl_gap = best_dl.cv_auc_mean - chosen_row.cv_auc_mean
n_dl_worse = int(((dl_val.gain_ci_high < 0) & (dl_val.p_adjusted_BH < 0.05)).sum())
dl_t = dl_test[dl_test.model.str.contains("neural")].set_index("model")["test AUC"]
dl_txt = "; ".join(f"{m.split(' (')[0]} {a:.2f}" for m, a in dl_t.items())
if n_dl_win:
    dl_head = f"In {n_dl_win} of {len(dl_val)} comparisons a network beat the best classic model with the same features on validation, so see the table. "
elif n_dl_worse:
    dl_head = f"No. In {n_dl_worse} of {len(dl_val)} comparisons the network was clearly worse than the best classic model with the same features (paired interval below 0, adjusted p < 0.05)" + ("; in the others it was not better. " if n_dl_worse < len(dl_val) else ". ")
else:
    dl_head = "No. The networks were not better than the classic models and not clearly worse either. "
lines.append(("2b. Do deep-learning models (MLP, GRU, 1D-CNN) do better?",
              dl_head
              + f"The best network on validation is {best_dl.network} on '{best_dl['feature group'].split(': ')[1]}' with mean AUC {best_dl.cv_auc_mean:.3f}, against {chosen_row.cv_auc_mean:.3f} for the chosen classic model ({dl_gap:+.3f}). "
              f"Supplementary test AUCs: {dl_txt}. With only {len(dev)} development movies and a 3-step page-view sequence the networks have little to learn from, which is the usual result on small tabular data. "
              "They also had fixed settings and no grid search, so part of the gap may come from less tuning. Adding Wikipedia or YouTube inputs made the networks worse, not better, which fits the main finding that pre-release buzz adds little. "
              "A CNN would need image or text inputs (posters, trailer frames, plot summaries) that are not in the two files."))
# Q2c
n_fam = len(leaderboard)
top_val = leaderboard.iloc[0]
top_test = leaderboard.sort_values("test AUC", ascending=False).iloc[0]
ci_w = (leaderboard.ci_high - leaderboard.ci_low).mean()
top6 = leaderboard.head(6)
conv = ex_val[ex_val.better_convincingly]
if len(conv):
    conv_txt = (f"{len(conv)} of {len(ex_val)} comparisons show an extra model that is convincingly better on validation than the best of Logistic Regression, Random Forest and HistGradientBoosting with the same features: "
                + "; ".join(f"{r.model} on '{r['feature group'].split(': ')[1]}' ({r['gain vs that model']:+.3f}, 95% CI {r.gain_ci_low:+.3f} to {r.gain_ci_high:+.3f}, adjusted p = {r.p_adjusted_BH:.3f})" for _, r in conv.iterrows())
                + ". The gains are small, and they compare two boosting libraries on the same features, not buzz against no buzz. ")
else:
    conv_txt = f"No extra model beat the best of Logistic Regression, Random Forest and HistGradientBoosting convincingly on validation (0 of {len(ex_val)} comparisons). "
lines.append(("2c. Do more model families change the picture?",
              f"We compared {n_fam} model families in total (3 main classic models, {len(set(ex_val.model))} extra ones including an average ensemble and a stacked model, and 3 neural networks). " + conv_txt
              + f"The highest validation AUC is {top_val['validation AUC']:.3f} ({top_val.model}); because so many models were tried, the best validation score is optimistic. "
              f"On the test period the top 6 families by validation AUC have test AUCs between {top6['test AUC'].min():.2f} and {top6['test AUC'].max():.2f}, a range "
              + ("smaller than" if top6['test AUC'].max() - top6['test AUC'].min() < ci_w else "similar to or larger than")
              + f" the typical confidence interval width ({ci_w:.2f}), so they are statistically " + ("tied" if top6['test AUC'].max() - top6['test AUC'].min() < ci_w else "different; check the intervals")
              + f"; the weakest families ({', '.join(leaderboard.tail(3).model)}) trail clearly on validation. The best test AUC overall is {top_test['test AUC']:.2f} ({top_test.model}), but that is a look at the test set, not a selection. "
              "The ceiling is set by the data (578 training movies, budget and language dominate), not by the choice of algorithm."))
# Q3
imp_main = imp_all[(CH_MODEL, list(GROUPS)[-1])] if (CH_MODEL, list(GROUPS)[-1]) in imp_all else imp_all[(CH_MODEL, CH_GROUP)]
g_imp = imp_main[imp_main.kind == "group"].sort_values("auc_drop", ascending=False)
f_imp = imp_main[imp_main.kind == "feature"].sort_values("auc_drop", ascending=False)
lines.append(("3. Which features matter most?",
              "By permutation importance on validation data, the information sources rank: " + ", ".join(f"{r.unit} ({r.auc_drop:.3f})" for _, r in g_imp.iterrows())
              + f". The single most important features are {', '.join(f_imp.unit.head(3))}. "
              + ("The top three sources are within one standard deviation of each other, so their order is not reliable. " if g_imp.auc_drop.iloc[0] - g_imp.auc_drop.iloc[min(2, len(g_imp) - 1)] < g_imp.sd.iloc[0] else "")
              + f"Importance is not the same as added value: the model does use the Wikipedia columns (mainly buzz_per_budget), yet adding Wikipedia buzz changes validation AUC by only {by_source.loc['Wikipedia', 'mean_gain']:+.3f} on average, "
              f"and Wikipedia buzz without the ratio changes it by {d_ab:+.3f} against base. buzz_per_budget partly repeats budget (linear overlap {results['buzz_per_budget_overlap_r2']:.0%}). "
              "Importance shows what the model uses, not what causes success."))


def tier(source_key, test_group):
    r = by_source.loc[source_key]
    cf = conf[conf["feature group"] == test_group].iloc[0]
    tg, tlo, thi = cf["gain vs base"], cf.ci_low, cf.ci_high
    if r.convincing == r.n_models and tlo > 0:
        label = "SUPPORTED"
    elif r.mean_gain >= 0.01 and tg > 0:
        label = "SUGGESTIVE but inconclusive"
    elif r.mean_gain <= 0 and tg <= 0:
        label = "NOT SUPPORTED"
    else:
        label = "INCONCLUSIVE (validation gain is about zero or disagrees with the test direction, and the test interval includes 0)"
    return (f"{label}. Validation: mean AUC gain {r.mean_gain:+.3f} across the 3 models, convincing in {int(r.convincing)} of {int(r.n_models)}. "
            f"Test confirmation ({CH_MODEL}): {tg:+.3f} (95% CI {tlo:+.3f} to {thi:+.3f}).")


lines.append(("4. Does Wikipedia buzz help?", tier("Wikipedia", list(GROUPS)[1])))
lines.append(("5. Does YouTube buzz help?", tier("YouTube", list(GROUPS)[2]) + " The YouTube result also depends on an unverified assumption (collected before release) and on partial coverage."))
both = cc_tab[cc_tab["complete-case subset"] == "both present"].iloc[0]
cc_note = (f" Exception to note: among the {int(both.movies)} movies that have both Wikipedia and YouTube data, the AUC gain is {both.gain:+.3f} (95% CI {both.ci_low:+.3f} to {both.ci_high:+.3f}, adjusted p = {both.p_adjusted_BH:.3f}). "
           "This is a post-hoc subgroup analysis defined by data availability (movies with both records are a selected, more notable group), so it is a hypothesis for future data, not a finding."
           if both.ci_low > 0 else "")
lines.append(("6. Does combining both help?", tier("Wikipedia + YouTube", list(GROUPS)[3]) + cc_note))
# Q7
lv = lang_test.dropna(subset=["AUC"])
lang_dev_rates = dev.groupby("language").hit.mean().sort_values()
lines.append(("7. How does performance differ across languages?",
              f"Hit rates differ a lot by language in the development data ({lang_dev_rates.index[0]} {lang_dev_rates.iloc[0]:.0%} to {lang_dev_rates.index[-1]} {lang_dev_rates.iloc[-1]:.0%}; chi-square significant after correction). The hit rate fell most in {lang_shift.change.idxmin()} ({lang_shift.loc[lang_shift.change.idxmin(), 'dev hit rate']:.0%} to {lang_shift.loc[lang_shift.change.idxmin(), 'test hit rate']:.0%}). "
              + (f"On test movies the AUC ranges from {lv.AUC.min():.2f} ({lv.loc[lv.AUC.idxmin(), 'language']}) to {lv.AUC.max():.2f} ({lv.loc[lv.AUC.idxmax(), 'language']}) for the languages with enough movies, with wide intervals; "
                 "languages with few movies (for example Kannada) cannot be evaluated reliably." if len(lv) else "Test movies per language are too few for a reliable language-wise AUC.")))
lines.append(("8. What are the major limitations?",
              f"Small sample ({len(lab)} usable movies; {len(test)} in the test period). Budget and revenue come from mixed public sources (TMDB and Wikipedia) with different ROI levels and no currency column (the infobox check in section 17g finds figures consistent with US dollars, but the gross/net basis is not stated). "
              f"The Hit rate shifted from {dev.hit.mean():.0%} to {test.hit.mean():.0%}. YouTube collection dates cannot be verified, YouTube coverage is partial and non-random, and 2026 releases have no YouTube data. "
              "Having a Wikipedia page or YouTube record may itself reflect later success. Revenue for some Indian films is incomplete on TMDB. The year feature cannot extrapolate to future years for tree models."))
ce = results["crew_experiment"]
ce_txt = (f"Cast and crew track records (director, writer, top cast, technical team, art director, production company and producer; computed only from films released at least {MATURITY_DAYS} days earlier) were tested as a separate experiment (section 17d): "
          f"{ce['convincing_gains']} of {ce['comparisons']} validation comparisons showed a convincing AUC gain (mean {ce['mean_val_gain']:+.3f}); on the test period the mean gain was {ce['mean_test_gain']:+.3f}. "
          + ("They are therefore not part of the main models." if ce["convincing_gains"] == 0 else "Because the gain is not confirmed everywhere, they are reported but not adopted in the main models."))
lines.append(("9. Do cast, crew, studio and producer track records help?", ce_txt))
cal_t = cal_table[cal_table.period == "test"].iloc[0]
lines.append(("10. Are the predicted probabilities calibrated?",
              f"On the test period the chosen model's average predicted Hit probability is {cal_t['mean predicted P(Hit)']:.2f} against an actual Hit rate of {cal_t['actual Hit rate']:.2f} (Brier {cal_t['Brier']:.3f}, "
              f"{'better' if cal_t['Brier skill'] > 0 else 'not better'} than always predicting the development Hit rate: skill {cal_t['Brier skill']:+.2f}). "
              "The probabilities are best used for ranking; for absolute probabilities they should be recalibrated on recent data (section 17e)."))
bf = recal_table[(recal_table.model == cal_models[0][0]) & (recal_table.recalibration != "none (as trained)")].sort_values("Brier").iloc[0]
lines.append(("11. Do recalibration, more data and the YouTube timing check change anything?",
              f"Recalibrating on recently labelled movies improves the probabilities only slightly: on the later half of the test period the best fix ('{bf.recalibration}') changes the Brier score by {bf['Brier gain vs none']:+.3f} (95% CI {bf['gain ci_low']:+.3f} to {bf['gain ci_high']:+.3f}) "
              f"and moves the average predicted Hit chance to {bf['mean predicted P(Hit)']:.2f} (actual {bf['actual Hit rate']:.2f}). The learning curve goes from {lc_main.loc[200]:.3f} AUC with 200 training movies to {lc_main.iloc[-1]:.3f} with {lc_main.index[-1]} (noisy, {len(va_last)} validation movies). "
              f"Likes per comment vs movie age: Spearman {rho_l:+.2f}; removing comment likes changes validation AUC by {yt_nolikes['change without likes (validation)'].mean():+.3f} on average."))
lines.append(("12. What should be investigated next?",
              "More movies with reliable budgets (the Wikipedia infoboxes of the unlabelled movies rarely state both budget and box office); verified collection timestamps for YouTube data; denser cast, crew and studio histories; search-trend and social-media data before release; "
              "a stated gross/net basis for every box-office figure; and a rolling re-training scheme tested on future releases."))
for q, a in lines:
    print(q)
    print("   " + a + "\n")
results["conclusions"] = [{"question": q, "answer": a} for q, a in lines]

print("Final model comparison (test period, one evaluation):")
display(final_table[["model", "feature group", "auc", "accuracy", "precision", "hit_recall", "flop_recall", "f1", "brier"]])

## 20. Limitations and future work
* **Small data.** A few hundred labelled movies make every comparison noisy; use the intervals, not the point estimates.
* **Label quality.** Budgets and revenues are public estimates from two sources with different ROI levels; the label is a proxy for profit (it ignores marketing cost and cinema share).
* **YouTube timing is unverified.** There is no collection date, so the YouTube features may include post-release information (especially comment likes).
* **Selection effects.** Only movies with a Wikipedia page or a YouTube record have buzz data, and those movies are not a random sample.
* **Market shift.** The Hit rate fell over time, so any model trained on the past will be over-optimistic about the future; retrain regularly.
* **Not causal.** None of the tests or importance measures show that a feature *causes* success.
* **Future work.** See the last conclusion.

## 21. Execution verification
The last cell checks, with assertions and not just printing, that every model trained, every figure was written, and `results.json` is complete and readable.

In [ ]:
EXPECTED_FIGS = ["01_label_and_time_trend.png", "02_correlations.png", "03_clusters.png", "04_buzz_gain_validation.png", "05_test_roc_pr_confusion.png", "06_test_confusion_matrices.png",
                 "07_validation_vs_test_curves.png", "09_shuffled_labels.png", "10_revenue_regression.png", "11_crew_track_records.png", "12_calibration.png", "14_recalibration.png", "15_learning_curve_and_youtube_timing.png"]
missing_figs = [f for f in EXPECTED_FIGS if not (OUT / f).exists()]
assert not missing_figs, f"missing figures: {missing_figs}"
assert set(results["final_test"]) == set(MODELS), "not all three classifiers were evaluated"
assert len(cv_table) == 1 + len(MODELS) * len(GROUPS), "validation table is incomplete"
for mk, v in results["final_test"].items():
    for k in ("auc", "accuracy", "precision", "hit_recall", "flop_recall", "f1", "brier"):
        assert v[k] == v[k], f"{mk}: metric {k} is NaN"
results["runtime_seconds"] = round(time.time() - T0)
results["environment"] = {"scikit_learn": sklearn.__version__, "pandas": pd.__version__, "numpy": np.__version__, "python": sys.version.split()[0]}
(OUT / "results.json").write_text(json.dumps(results, indent=2, default=lambda o: o.item() if isinstance(o, np.generic) else (o.tolist() if isinstance(o, np.ndarray) else str(o))))
reloaded = json.loads((OUT / "results.json").read_text())
print(f"results.json written with {len(reloaded)} top-level sections; {len(list(OUT.glob('*.png')))} figures in {OUT}/.")
print(f"All checks passed. Total runtime {results['runtime_seconds']} s.")

## 22. Download the results as CSV files
This cell saves every main table as a CSV file in the `outputs/` folder, together with the figures and `results.json`, and packs everything into one zip file.
In Colab the zip downloads automatically. **Colab deletes the `outputs/` folder when the session ends, so download the zip before closing the notebook.**

In [ ]:
import shutil
TABLES = {
    "data_audit": audit, "filter_log": filter_log, "feature_audit": feature_audit, "outer_folds": fold_table,
    "hypothesis_tests": tt, "hit_rate_by_language_dev": lang.reset_index(), "spearman_vs_log_roi": sp, "cluster_profiles": prof.reset_index(),
    "validation_results": cv_table.drop(columns=["fold_aucs"]), "buzz_gain_validation": pairs, "buzz_gain_by_source": by_source.reset_index(), "model_vs_model_validation": model_pairs,
    "final_test_results": final_table, "threshold_options": thr_table, "buzz_confirmation_on_test": conf,
    "robust_roi_thresholds": roi_tab, "robust_complete_case": cc_tab, "robust_language_shift": lang_shift.reset_index(), "robust_sensitivity": sens_tab,
    "extra_models_validation": ex_val.drop(columns=["worse_convincingly"]), "extra_models_test": ex_test, "all_models_leaderboard": leaderboard, "deep_learning_validation": dl_val, "deep_learning_test": dl_test, "regression_validation": reg_cv, "regression_test": reg_test, "permutation_importance": imp_main,
    "crew_experiment_validation": crew_val, "crew_experiment_test": crew_test, "calibration": cal_table, "shap_importance": shap_table,
    "recalibration_test_halves": recal_table, "currency_check": currency_check, "learning_curve": lc_table, "youtube_timing_by_year": yt_year_table, "youtube_without_comment_likes": yt_nolikes,
}
saved = []
for name, table in TABLES.items():
    table.to_csv(OUT / f"{name}.csv", index=False)
    saved.append(name + ".csv")
# the cleaned, labelled movies used for modelling (development and test period), without free-text columns
lab_export = lab[["tmdb_id", "name", "year", "release_date", "language", "budget", "revenue", "roi", "hit"]].assign(period=np.where(lab.release_date < SPLIT, "development", "test"))
lab_export.to_csv(OUT / "labelled_movies.csv", index=False)
saved.append("labelled_movies.csv")
assert all((OUT / s).stat().st_size > 0 for s in saved), "an exported CSV is empty"
zip_path = shutil.make_archive("movie_hit_flop_results", "zip", OUT)
print(f"Saved {len(saved)} CSV files in {OUT}/ and packed outputs/ into {zip_path}:")
print(", ".join(saved))
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print("Not running in Colab: take the zip from the notebook folder.")